In [ ]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import cupy as cp
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score, confusion_matrix, mean_absolute_error
)
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression

# 修改：正确导入cuML的相关模块
from cuml.ensemble import RandomForestClassifier as cuRF, RandomForestRegressor as cuRFR
from cuml.linear_model import LogisticRegression as cuLogisticRegression, LinearRegression as cuLinearRegression
from cuml.svm import SVC as cuSVC, SVR as cuSVR
from cuml.model_selection import train_test_split as cu_train_test_split  # 修正导入
from cuml.metrics import accuracy_score as cu_accuracy_score
from cuml import __version__ as cuml_version
import warnings
warnings.filterwarnings("ignore", message="Random state is currently ignored by probabilistic SVC")

# 新增：梯度提升模型导入
try:
    from cuml.ensemble import GradientBoostingClassifier as cuGBC, GradientBoostingRegressor as cuGBR
    HAS_GRADIENT_BOOSTING = True
except ImportError:
    print("cuML Gradient Boosting 不可用，将跳过该模型")
    HAS_GRADIENT_BOOSTING = False

# XGBoost导入
import xgboost as xgb

# 其他导入
from datetime import datetime
from pandas.api.types import CategoricalDtype
import traceback
from sklearn.base import clone, BaseEstimator
from sklearn.model_selection import KFold
from typing import Dict, Any, Tuple, Optional, List
import warnings
import json
warnings.filterwarnings('ignore')

# 新增：用于生成Word文档的库
from docx import Document
from docx.shared import Inches

# 新增：用于深度复制PyTorch模型
import copy

# 设置环境变量
os.environ['XGBOOST_DEVICE_MEMORY_LIMIT'] = '10GB'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# ========================
# GPU资源清理增强模块
# ========================
def clear_gpu_resources():
    """全面清理GPU资源并添加延迟"""
    gc.collect()
    time.sleep(0.5)
    
    if 'cp' in globals():
        try:
            cp._default_memory_pool.free_all_blocks()
            cp.get_default_memory_pool().free_all_blocks()
        except Exception: 
            pass
        
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass

def monitor_gpu_memory(threshold=0.8):
    """监控GPU内存使用，超过阈值则清理"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated()
        total = torch.cuda.get_device_properties(0).total_memory
        ratio = allocated / total
        if ratio > threshold:
            print(f"GPU内存使用率超过{threshold*100:.1f}% ({allocated/1e9:.2f}GB/{total/1e9:.2f}GB)，执行清理...")
            clear_gpu_resources()
            return True
    return False

# ========================
# 辅助函数：PyTorch版本的自助法标准误
# ========================
def bootstrap_se_torch(X: np.ndarray, y: np.ndarray, n_bootstrap: int = 1000) -> Tuple[float, float, float]:
    """
    使用自助法计算标准误和置信区间（PyTorch优化版）
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_samples = len(y)
    
    bootstrap_effects = []
    
    for i in range(n_bootstrap):
        # 生成自助样本
        indices = np.random.choice(n_samples, n_samples, replace=True)
        X_boot = torch.tensor(X[indices], dtype=torch.float32, device=device)
        y_boot = torch.tensor(y[indices], dtype=torch.float32, device=device)
        
        # 使用正规方程求解
        with torch.no_grad():
            XtX = X_boot.t() @ X_boot
            Xty = X_boot.t() @ y_boot.reshape(-1, 1)
            
            # 添加小常数防止奇异矩阵
            XtX += torch.eye(X_boot.shape[1], device=device) * 1e-6
            
            coefficients = torch.linalg.solve(XtX, Xty)
            effect = coefficients[1].item() if coefficients.shape[0] > 1 else coefficients[0].item()
        
        bootstrap_effects.append(effect)
    
    # 计算标准误和置信区间
    bootstrap_effects = np.array(bootstrap_effects)
    se = np.std(bootstrap_effects, ddof=1)
    ci_lower = np.percentile(bootstrap_effects, 2.5)
    ci_upper = np.percentile(bootstrap_effects, 97.5)
    
    return se, ci_lower, ci_upper

# ========================
# 辅助函数：PyTorch版本的聚类稳健标准误
# ========================
def calculate_cluster_robust_se_torch(X: torch.Tensor, 
                                     y: torch.Tensor, 
                                     residuals: torch.Tensor, 
                                     cluster_ids: np.ndarray) -> float:
    """
    计算聚类稳健标准误（PyTorch版本）
    """
    device = X.device
    unique_clusters = np.unique(cluster_ids)
    n_clusters = len(unique_clusters)
    
    scores = []
    
    for cluster in unique_clusters:
        cluster_mask = (cluster_ids == cluster)
        X_cluster = X[cluster_mask]
        residuals_cluster = residuals[cluster_mask]
        
        # 计算聚类得分
        cluster_score = X_cluster.t() @ residuals_cluster
        scores.append(cluster_score.cpu().numpy())
    
    # 计算聚类稳健方差估计
    scores_matrix = np.array(scores)
    mean_score = np.mean(scores_matrix, axis=0)
    cluster_variance = (scores_matrix - mean_score).T @ (scores_matrix - mean_score) / (n_clusters - 1)
    
    # 计算标准误
    XtX_inv = torch.linalg.inv(X.t() @ X).cpu().numpy()
    cluster_vcov = XtX_inv @ cluster_variance @ XtX_inv
    se = np.sqrt(np.diag(cluster_vcov))[1]
    
    return se

# ========================
# 新增：第一阶段模型工厂类,修改MLP实现
# ========================
class FirstStageModelFactory:
    """第一阶段模型工厂类，用于创建和配置不同的机器学习模型"""
    def __init__(self):
        self.model_configs = {
            'RandomForest': {
                'classification': lambda: cuRF(n_estimators=200, max_depth=20, random_state=42),
                'regression': lambda: cuRFR(n_estimators=200, max_depth=20, random_state=42)
            },
            'LogisticRegression': {
                'classification': lambda: cuLogisticRegression(penalty='l2', C=1.0, max_iter=1000),  # 移除random_state
                'regression': lambda: cuLinearRegression(fit_intercept=True, algorithm='eig')
            },
            'GradientBoosting': {
                'classification': lambda: cuGBC(n_estimators=200, max_depth=6, random_state=42) if HAS_GRADIENT_BOOSTING else None,
                'regression': lambda: cuGBR(n_estimators=200, max_depth=6, random_state=42) if HAS_GRADIENT_BOOSTING else None
            },
            'SVC': {
                'classification': lambda: cuSVC(kernel='rbf', C=1.0, probability=True, random_state=42),
                'regression': lambda: cuSVR(kernel='rbf', C=1.0)
            },
            'XGBoost': {
                'classification': lambda: xgb.XGBClassifier(n_estimators=200, max_depth=6, random_state=42, tree_method='gpu_hist'),
                'regression': lambda: xgb.XGBRegressor(n_estimators=200, max_depth=6, random_state=42, tree_method='gpu_hist')
            },
            'MLP': {
                'classification': lambda input_size: PyTorchMLP(input_size=input_size, output_size=1, task_type='classification'),
                'regression': lambda input_size: PyTorchMLP(input_size=input_size, output_size=1, task_type='regression')
            }
        }
    
    def get_model(self, model_name, task_type, input_size=None):
        """
        根据模型名称和任务类型获取模型实例
        """
        if model_name not in self.model_configs:
            raise ValueError(f"不支持的模型类型: {model_name}")
        
        if task_type not in self.model_configs[model_name]:
            raise ValueError(f"模型 {model_name} 不支持任务类型: {task_type}")
        
        # 特殊处理MLP模型，需要输入维度
        if model_name == 'MLP':
            if input_size is None:
                raise ValueError("MLP模型需要指定input_size参数")
            return self.model_configs[model_name][task_type](input_size)
        else:
            model = self.model_configs[model_name][task_type]()
            if model is None:
                raise ValueError(f"模型 {model_name} 不可用，可能缺少依赖")
            return model

# ========================
# 新增：PyTorch MLP模型（GPU版本）修改
# ========================
class PyTorchMLP(nn.Module):
    """PyTorch多层感知机模型（GPU加速版），用于替代cuML的MLP"""

    def __init__(self, input_size, hidden_sizes=[100, 50], output_size=1, 
                 task_type='classification', dropout_rate=0.1):
        super(PyTorchMLP, self).__init__()
        self.task_type = task_type
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # 构建网络层
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, output_size))
        
        # 根据任务类型添加输出层激活函数
        if task_type == 'classification':
            layers.append(nn.Sigmoid())
        
        self.network = nn.Sequential(*layers).to(self.device)
        
    def forward(self, x):
        return self.network(x)
    
    def fit(self, X, y, epochs=100, batch_size=256, lr=0.001):
        """训练模型"""
        self.train()
        
        # 转换数据为PyTorch张量
        if not isinstance(X, torch.Tensor):
            X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        else:
            X_tensor = X.to(self.device)
            
        if not isinstance(y, torch.Tensor):
            y_tensor = torch.tensor(y, dtype=torch.float32, device=self.device)
        else:
            y_tensor = y.to(self.device)
        
        # 确保输出维度匹配
        if self.task_type == 'classification':
            y_tensor = y_tensor.view(-1, 1)
        
        # 定义损失函数和优化器
        if self.task_type == 'classification':
            criterion = nn.BCELoss()
        else:
            criterion = nn.MSELoss()
        
        optimizer = optim.Adam(self.parameters(), lr=lr)
        
        # 训练循环
        dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
        
        for epoch in range(epochs):
            epoch_loss = 0.0
            for batch_X, batch_y in dataloader:
                optimizer.zero_grad()
                outputs = self(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            if (epoch + 1) % 20 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(dataloader):.4f}")
    
    def predict(self, X):
        """预测"""
        self.eval()
        with torch.no_grad():
            if not isinstance(X, torch.Tensor):
                X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            else:
                X_tensor = X.to(self.device)
                
            predictions = self(X_tensor)
            
            if self.task_type == 'classification':
                return (predictions > 0.5).float().cpu().numpy().flatten()
            else:
                return predictions.cpu().numpy().flatten()
    
    def predict_proba(self, X):
        """预测概率（仅分类任务）"""
        if self.task_type != 'classification':
            raise ValueError("predict_proba仅适用于分类任务")
        
        self.eval()
        with torch.no_grad():
            if not isinstance(X, torch.Tensor):
                X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            else:
                X_tensor = X.to(self.device)
                
            return self(X_tensor).cpu().numpy()

# ========================
# 新增：使用二折交叉验证计算残差的辅助函数
# ========================
def get_y_residuals(model_proto, X, y, n_splits=5):
    """
    使用交叉验证计算outcome (y) 的残差
    对于y，使用predict (标签) 计算残差
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    residuals = np.zeros(len(y))
    
    for train_idx, test_idx in kf.split(X):
        # 克隆模型
        if isinstance(model_proto, PyTorchMLP):
            model_clone = copy.deepcopy(model_proto)
        else:
            model_clone = clone(model_proto)
        
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # 训练克隆模型
        model_clone.fit(X_train, y_train)
        
        # 预测 (使用predict，得到标签)
        y_pred = model_clone.predict(X_test)
        
        # 计算残差
        residuals[test_idx] = y_test - y_pred
    
    return residuals

def get_t_residuals(model_proto, X, t, n_splits=5):
    """
    使用交叉验证计算treatment (t) 的残差
    对于t，如果有predict_proba，使用概率；否则使用predict
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    residuals = np.zeros(len(t))
    
    for train_idx, test_idx in kf.split(X):
        # 克隆模型
        if isinstance(model_proto, PyTorchMLP):
            model_clone = copy.deepcopy(model_proto)
        else:
            model_clone = clone(model_proto)
        
        X_train, X_test = X[train_idx], X[test_idx]
        t_train, t_test = t[train_idx], t[test_idx]
        
        # 训练克隆模型
        model_clone.fit(X_train, t_train)
        
        # 预测
        if hasattr(model_clone, 'predict_proba'):
            t_pred = model_clone.predict_proba(X_test)[:, 1] if model_clone.predict_proba(X_test).ndim > 1 else model_clone.predict_proba(X_test)
        else:
            t_pred = model_clone.predict(X_test)
        
        # 计算残差
        residuals[test_idx] = t_test - t_pred
    
    return residuals

# ========================
# 使用PyTorch进行DML分析（优化版）
# ========================
from scipy import stats
def second_stage_dml_analysis(t_residuals, y_residuals, X_second=None, robust_method='bootstrap', n_bootstrap=1000):
    """
    执行DML的第二阶段分析，计算处理效应及统计检验指标。
    
    参数:
        t_residuals (torch.Tensor或np.array): 第一阶段处理模型的残差。
        y_residuals (torch.Tensor或np.array): 第一阶段结果模型的残差。
        X_second (torch.Tensor或np.array, optional): 用于异质性分析的协变量。默认为None。
        robust_method (str): 稳健标准误方法，'bootstrap'或'hc1'。默认为'bootstrap'。
        n_bootstrap (int): Bootstrap重复次数，仅当robust_method='bootstrap'时有效。默认为1000。
    
    返回:
        dict: 包含效应值、标准误、置信区间、t统计量、p值、F统计量、F检验p值等的字典。
    """
    # 确保将Tensor转换为NumPy数组
    if torch.is_tensor(t_residuals):
        t_residuals = t_residuals.cpu().numpy() if t_residuals.is_cuda else t_residuals.numpy()
    if torch.is_tensor(y_residuals):
        y_residuals = y_residuals.cpu().numpy() if y_residuals.is_cuda else y_residuals.numpy()
    if X_second is not None and torch.is_tensor(X_second):
        X_second = X_second.cpu().numpy() if X_second.is_cuda else X_second.numpy()
    
    n = len(y_residuals)
    
    # 确保残差是NumPy数组并调整为正确的形状
    t_residuals = np.asarray(t_residuals).reshape(-1, 1)
    y_residuals = np.asarray(y_residuals).reshape(-1, 1)
    
    # 第二阶段回归：Y_residuals ~ theta * T_residuals + [其他交互项]
    if X_second is not None:
        # 如果有协变量用于异质性分析（CATE），包含交互项
        X_second = np.asarray(X_second)
        # 构建设计矩阵: [T_residuals, T_residuals * X_second]
        interaction_terms = t_residuals * X_second
        design_matrix = np.column_stack((t_residuals, interaction_terms))
    else:
        # 只估计ATE
        design_matrix = t_residuals
    
    # 使用OLS估计系数
    coefficients, _, _, _ = np.linalg.lstsq(design_matrix, y_residuals, rcond=None)
    
    # 提取处理效应（ATE是第一个系数）
    effect = coefficients[0][0] if len(coefficients) > 0 else 0.0
    
    # 计算预测值和残差
    y_pred = np.dot(design_matrix, coefficients)
    residuals = y_residuals - y_pred
    
    # 初始化变量
    se = 0.0
    t_statistic = 0.0
    p_value = 1.0
    f_statistic = 0.0
    f_p_value = 1.0
    
    # 计算标准误和置信区间
    if robust_method == 'bootstrap':
        bootstrap_effects = []
        for _ in range(n_bootstrap):
            indices = np.random.choice(n, n, replace=True)
            t_boot = t_residuals[indices]
            y_boot = y_residuals[indices]
            if X_second is not None:
                X_second_boot = X_second[indices]
                interaction_boot = t_boot * X_second_boot
                design_boot = np.column_stack((t_boot, interaction_boot))
            else:
                design_boot = t_boot
            coef_boot, _, _, _ = np.linalg.lstsq(design_boot, y_boot, rcond=None)
            bootstrap_effects.append(coef_boot[0][0] if len(coef_boot) > 0 else 0.0)
        se = np.std(bootstrap_effects, ddof=1) if bootstrap_effects else 0.0
    else:
        # 使用HC1稳健标准误的简化实现
        mse = np.sum(residuals**2) / (n - design_matrix.shape[1])
        xtx_inv = np.linalg.inv(np.dot(design_matrix.T, design_matrix))
        se_matrix = mse * xtx_inv
        se = np.sqrt(np.diag(se_matrix)[0]) if se_matrix.size > 0 else 0.0
    
    # 计算置信区间
    ci_lower = effect - 1.96 * se if se > 0 else effect
    ci_upper = effect + 1.96 * se if se > 0 else effect
    
    # 计算t统计量和p值 (只有在se>0时才有意义)
    if se > 0:
        t_statistic = effect / se
        df = n - design_matrix.shape[1]  # 自由度
        p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))
    
    # 计算F统计量和p值 (检验整个第二阶段模型是否显著)
    ss_total = np.sum((y_residuals - np.mean(y_residuals))**2)
    ss_residual = np.sum(residuals**2)
    if ss_residual > 0 and design_matrix.shape[1] > 0:
        ss_explained = ss_total - ss_residual
        df_model = design_matrix.shape[1]
        df_resid = n - df_model
        f_statistic = (ss_explained / df_model) / (ss_residual / df_resid) if df_resid > 0 else 0.0
        f_p_value = 1 - stats.f.cdf(f_statistic, df_model, df_resid) if df_resid > 0 else 1.0
    
    # 返回所有结果
    return {
        'effect': effect,
        'se': se,
        'lb': ci_lower,
        'ub': ci_upper,
        't_statistic': t_statistic,
        'p_value': p_value,
        'f_statistic': f_statistic,
        'f_p_value': f_p_value,
        'df': n - design_matrix.shape[1] if design_matrix.shape[1] > 0 else n
    }

def torch_dml_with_pretrained(X: np.ndarray, 
                              y: np.ndarray, 
                              t: np.ndarray, 
                              model_y: BaseEstimator, 
                              model_t: BaseEstimator, 
                              use_robust_se: bool = True, 
                              robust_method: str = 'bootstrap',
                              n_bootstrap: int = 1000,
                              cluster_ids: Optional[np.ndarray] = None,
                              n_splits: int = 5) -> Dict[str, Any]:
    """
    使用预训练的第一阶段模型执行DML的第二阶段分析（残差回归）
    
    参数:
        X: 协变量矩阵
        y: 结果变量
        t: 处理变量
        model_y: 预训练的结果变量模型
        model_t: 预训练的处理变量模型
        use_robust_se: 是否使用稳健标准误
        robust_method: 稳健标准误计算方法 ('bootstrap' 或 'hc1')
        n_bootstrap: 自助法重采样次数
        cluster_ids: 聚类ID（用于聚类稳健标准误）
        n_splits: 交叉验证折数
    
    返回:
        包含效应值、标准误、置信区间等结果的字典
    """
    print("执行DML第二阶段分析（残差回归）...")
    
    # 初始化所有变量
    effect = 0.0
    se = 0.0
    ci_lower = 0.0
    ci_upper = 0.0
    t_statistic = 0.0
    p_value = 1.0
    f_statistic = 0.0
    f_p_value = 1.0
    df = 0
    
    # 确保输入为NumPy数组
    X_np = X if isinstance(X, np.ndarray) else cp.asnumpy(X)
    y_np = y if isinstance(y, np.ndarray) else cp.asnumpy(y)
    t_np = t if isinstance(t, np.ndarray) else cp.asnumpy(t)
    
    # 第一阶段：使用指定折数交叉验证计算残差
    print(f"使用{n_splits}折交叉验证计算第一阶段残差...")
    y_residuals = get_y_residuals(model_y, X_np, y_np, n_splits=n_splits)
    t_residuals = get_t_residuals(model_t, X_np, t_np, n_splits=n_splits)
    
    # 转换为PyTorch张量
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    t_residuals_tensor = torch.tensor(t_residuals, dtype=torch.float32, device=device).reshape(-1, 1)
    y_residuals_tensor = torch.tensor(y_residuals, dtype=torch.float32, device=device)
    
    # 添加截距项
    ones = torch.ones_like(t_residuals_tensor, device=device)
    X_second = torch.cat([ones, t_residuals_tensor], dim=1)
    
    # 使用正规方程求解第二阶段系数
    with torch.no_grad():
        XtX = X_second.t() @ X_second
        Xty = X_second.t() @ y_residuals_tensor.reshape(-1, 1)
        
        # 添加小常数防止奇异矩阵
        XtX += torch.eye(2, device=device) * 1e-6
        
        coefficients = torch.linalg.solve(XtX, Xty)
        effect = coefficients[1].item()  # 处理效应系数
    
    # 计算标准误和置信区间
    if use_robust_se and robust_method == 'bootstrap':
        print("使用自助法计算稳健标准误...")
        se, ci_lower, ci_upper = bootstrap_se_torch(
            X_second.cpu().numpy(), 
            y_residuals_tensor.cpu().numpy(), 
            n_bootstrap=n_bootstrap
        )
        
        # 计算t统计量和p值
        if se > 0:
            t_statistic = effect / se
            n, k = X_second.shape
            df = n - k
            p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))
    else:
        # 计算经典标准误
        n, k = X_second.shape
        with torch.no_grad():
            y_pred_tensor = X_second @ coefficients
            residuals = y_residuals_tensor.reshape(-1, 1) - y_pred_tensor
        mse = torch.sum(residuals**2) / (n - k)
        cov_matrix = torch.linalg.inv(XtX) * mse
        se = torch.sqrt(torch.diag(cov_matrix))[1].item()
        
        # 计算置信区间
        t_stat_val = 1.96  # 95%置信水平的t统计量
        ci_lower = effect - t_stat_val * se
        ci_upper = effect + t_stat_val * se
        
        # 计算t统计量
        t_statistic = effect / se

        # 计算p值（双侧检验）
        df = n - k  # 自由度
        p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))

        # 计算F统计量
        with torch.no_grad():
            y_pred_tensor = X_second @ coefficients
            ss_residual = torch.sum((y_residuals_tensor.reshape(-1, 1) - y_pred_tensor)**2)
            ss_total = torch.sum((y_residuals_tensor.reshape(-1, 1) - torch.mean(y_residuals_tensor))**2)
            ss_explained = ss_total - ss_residual
    
            # F统计量 = (解释方差/自由度) / (残差方差/自由度)
            f_statistic = (ss_explained / (k - 1)) / (ss_residual / (n - k))
            f_p_value = 1 - stats.f.cdf(f_statistic, k - 1, n - k)
    
    print(f"DML处理效应估计: {effect:.6f}")
    print(f"标准误: {se:.6f}")
    print(f"95%置信区间: [{ci_lower:.6f}, {ci_upper:.6f}]")
    print(f"t统计量: {t_statistic:.4f}")
    print(f"p值: {p_value:.6f}")
    print(f"F统计量: {f_statistic:.4f}")
    print(f"F检验p值: {f_p_value:.6f}")
    
    return {
        'effect': effect,
        'se': se,
        'lb': ci_lower,
        'ub': ci_upper,
        't_statistic': t_statistic,
        'p_value': p_value,
        'f_statistic': f_statistic,
        'f_p_value': f_p_value,
        'df': df
    }

# ========================
# 修改：增强的第一阶段训练函数，尽量使用cuml
# ========================
def first_stage_cuml_enhanced(X, y, task_type='regression', model_name='RandomForest', test_size=0.25):
    """
    增强版第一阶段预测，支持多种模型 - 使用新的评估函数
    """
    clear_gpu_resources()
    
    # 数据分割 - 使用cuML的train_test_split
    X_train, X_test, y_train, y_test = cu_train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    
    # 创建模型
    model_factory = FirstStageModelFactory()
    
    try:
        # 特殊处理MLP模型，需要输入维度
        if model_name == 'MLP':
            input_size = X_train.shape[1]
            model = model_factory.get_model(model_name, task_type, input_size)
        else:
            model = model_factory.get_model(model_name, task_type)
        
        # 特殊处理PyTorch MLP模型
        if model_name == 'MLP':
            # 转换为PyTorch张量
            X_train_pt = torch.tensor(cp.asnumpy(X_train), dtype=torch.float32)
            y_train_pt = torch.tensor(cp.asnumpy(y_train), dtype=torch.float32)
            
            # 训练模型
            model.fit(X_train_pt, y_train_pt)
            
            # 预测
            X_test_pt = torch.tensor(cp.asnumpy(X_test), dtype=torch.float32)
            y_pred = model.predict(X_test_pt)
            y_pred = cp.array(y_pred)
        else:
            # 训练其他cuML/XGBoost模型
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
        
        # 使用新的评估函数计算评估指标 - 添加task_type参数
        metrics = evaluate_model(model, X_test, y_test, model_name, task_type)
        
        # 添加模型信息
        metrics['model_name'] = model_name
        metrics['task_type'] = task_type
        
        # 根据任务类型输出结果
        if task_type == 'classification':
            summary = f"准确率: {metrics.get('accuracy', 0):.4f}, F1: {metrics.get('f1', 0):.4f}, AUC: {metrics.get('auc', 0):.4f}"
        else:
            summary = f"R²: {metrics.get('r2', 0):.4f}, RMSE: {metrics.get('rmse', 0):.4f}, MAE: {metrics.get('mae', 0):.4f}"
        
        print(f"{model_name} {task_type}模型评估 - {summary}")
        
        return model, metrics
        
    except Exception as e:
        print(f"训练{model_name}模型失败: {str(e)}")
        traceback.print_exc()
        return None, None

# ========================
# 新增：评估指标计算函数
# ========================
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
def evaluate_model(model, X_test, y_test, model_name="Model", task_type='classification'):
    """
    评估模型性能，根据任务类型使用不同的指标
    
    参数:
        model: 训练好的模型
        X_test: 测试特征
        y_test: 测试标签
        model_name: 模型名称
        task_type: 任务类型 ('regression' 或 'classification')
    
    返回:
        dict: 包含评估指标的字典
    """
    try:
        # 确保数据是NumPy数组（解决"Implicit conversion to a NumPy array is not allowed"错误）
        if hasattr(X_test, 'get'):
            X_test_np = X_test.get()
        elif hasattr(X_test, 'cpu'):
            X_test_np = X_test.cpu().numpy()
        elif hasattr(X_test, 'numpy'):
            X_test_np = X_test.numpy()
        else:
            X_test_np = np.array(X_test)
            
        if hasattr(y_test, 'get'):
            y_test_np = y_test.get()
        elif hasattr(y_test, 'cpu'):
            y_test_np = y_test.cpu().numpy()
        elif hasattr(y_test, 'numpy'):
            y_test_np = y_test.numpy()
        else:
            y_test_np = np.array(y_test)
        
        # 预测
        y_pred = model.predict(X_test_np)
        
        # 确保预测结果也是NumPy数组
        if hasattr(y_pred, 'get'):
            y_pred_np = y_pred.get()
        elif hasattr(y_pred, 'cpu'):
            y_pred_np = y_pred.cpu().numpy()
        elif hasattr(y_pred, 'numpy'):
            y_pred_np = y_pred.numpy()
        else:
            y_pred_np = np.array(y_pred)
        
        metrics = {}
        
        if task_type == 'regression':
            # 回归任务评估指标
            metrics['r2'] = r2_score(y_test_np, y_pred_np)
            metrics['mse'] = mean_squared_error(y_test_np, y_pred_np)
            metrics['rmse'] = np.sqrt(metrics['mse'])
            metrics['mae'] = mean_absolute_error(y_test_np, y_pred_np)
            
            print(f"{model_name} 回归模型评估 - R²: {metrics['r2']:.4f}, RMSE: {metrics['rmse']:.4f}, MAE: {metrics['mae']:.4f}")
            
        else:
            # 分类任务评估指标
            metrics['accuracy'] = accuracy_score(y_test_np, y_pred_np)
            metrics['precision'] = precision_score(y_test_np, y_pred_np, average='weighted')
            metrics['recall'] = recall_score(y_test_np, y_pred_np, average='weighted')
            metrics['f1'] = f1_score(y_test_np, y_pred_np, average='weighted')
            
            # 计算AUC
            auc_score = np.nan
            try:
                if hasattr(model, 'predict_proba'):
                    y_proba = model.predict_proba(X_test_np)
                    if hasattr(y_proba, 'get'):
                        y_proba = y_proba.get()
                    elif hasattr(y_proba, 'cpu'):
                        y_proba = y_proba.cpu().numpy()
                    
                    if len(y_proba.shape) > 1 and y_proba.shape[1] == 2:  # 二分类
                        auc_score = roc_auc_score(y_test_np, y_proba[:, 1])
                    elif len(y_proba.shape) > 1:  # 多分类
                        auc_score = roc_auc_score(y_test_np, y_proba, multi_class='ovr')
                    else:  # 一维数组
                        auc_score = roc_auc_score(y_test_np, y_proba)
                elif hasattr(model, 'decision_function'):
                    y_score = model.decision_function(X_test_np)
                    if hasattr(y_score, 'get'):
                        y_score = y_score.get()
                    elif hasattr(y_score, 'cpu'):
                        y_score = y_score.cpu().numpy()
                    auc_score = roc_auc_score(y_test_np, y_score)
                else:
                    print(f"警告: 模型 {model_name} 不支持概率预测，无法计算AUC")
                    auc_score = np.nan
            except Exception as e:
                print(f"警告: 计算模型 {model_name} 的AUC时出错: {str(e)}")
                auc_score = np.nan
            
            metrics['auc'] = auc_score
            print(f"{model_name} 分类模型评估 - 准确率: {metrics['accuracy']:.4f}, 精确率: {metrics['precision']:.4f}, 召回率: {metrics['recall']:.4f}, F1: {metrics['f1']:.4f}, AUC: {metrics['auc']:.4f}")
        
        metrics['predictions'] = y_pred_np
        return metrics
        
    except Exception as e:
        print(f"评估模型 {model_name} 时出错: {str(e)}")
        traceback.print_exc()
        if task_type == 'regression':
            return {
                'r2': np.nan,
                'mse': np.nan,
                'rmse': np.nan,
                'mae': np.nan,
                'predictions': None
            }
        else:
            return {
                'accuracy': np.nan,
                'precision': np.nan,
                'recall': np.nan,
                'f1': np.nan,
                'auc': np.nan,
                'predictions': None
            }

# ========================
# 新增：模型比较器类
# ========================
class ModelComparator:
    """模型比较器，用于比较不同模型的性能并选择最佳模型"""
    
    def __init__(self, task_type):
        self.task_type = task_type
        self.model_metrics = {}
        self.weights = self._get_default_weights()
    
    def add_model(self, model_name, metrics):
        """添加模型及其评估指标"""
        self.model_metrics[model_name] = metrics
    
    def _get_default_weights(self):
        """获取默认指标权重"""
        if self.task_type == 'classification':
            return {'accuracy': 0.2, 'precision': 0.2, 'recall': 0.2, 'f1': 0.2, 'auc': 0.2}
        else:
            return {'r2': 0.4, 'rmse': 0.3, 'mae': 0.3}
    
    def _calculate_score(self, metrics):
        """计算模型综合得分（修复版）"""
        if self.task_type == 'regression':
            # 以R²为主要依据，同时考虑RMSE和MAE（数值越小越好，故取倒数）
            r2 = metrics.get('r2', 0)
            rmse = metrics.get('rmse', 1e6)
            mae = metrics.get('mae', 1e6)
            # 如果R²为负，给予惩罚性低分
            if r2 < 0:
                return -1.0 * abs(r2) # 或 return 0.0
            # 综合得分，R²权重最高
            score = r2 * 0.7 + (1 / (rmse + 1e-6)) * 0.15 + (1 / (mae + 1e-6)) * 0.15
            return score
        else:
            # 分类任务：可以考虑准确率、F1、AUC的加权平均
            accuracy = metrics.get('accuracy', 0)
            precision = metrics.get('precision', 0)
            recall = metrics.get('recall', 0)
            f1 = metrics.get('f1', 0)
            auc = metrics.get('auc', 0)
            if np.isnan(auc):
                auc = 0 # 将NaN的AUC视为0
            # 赋予AUC较高权重，如果AUC为0则主要依赖准确率和F1
            if auc > 0:
                score = accuracy * 0.2 + precision * 0.2 + recall * 0.2 + f1 * 0.2 + auc * 0.2
            else:
                score = accuracy * 0.25 + precision * 0.25 + recall * 0.25 + f1 * 0.25
            return score
    
    def compare_models(self):
        """比较所有模型并返回排名"""
        model_scores = {}
        
        for model_name, metrics in self.model_metrics.items():
            score = self._calculate_score(metrics)
            model_scores[model_name] = score
        
        # 按得分排序
        sorted_models = sorted(model_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_models
    
    def get_best_model(self):
        """获取最佳模型名称"""
        sorted_models = self.compare_models()
        return sorted_models[0][0] if sorted_models else None
    
    def print_comparison(self):
        """打印模型比较结果 - 使用新的评估函数"""
        print(f"\n=== {self.task_type.upper()} 模型比较结果 ===")
        sorted_models = self.compare_models()
        
        for rank, (model_name, score) in enumerate(sorted_models, 1):
            metrics = self.model_metrics[model_name]
            
            # 使用新的评估函数格式输出
            if self.task_type == 'classification':
                summary = f"准确率: {metrics.get('accuracy', 0):.4f}, 精确率: {metrics.get('precision', 0):.4f}, 召回率: {metrics.get('recall', 0):.4f}, F1: {metrics.get('f1', 0):.4f}, AUC: {metrics.get('auc', 0):.4f}"
            else:
                summary = f"R²: {metrics.get('r2', 0):.4f}, RMSE: {metrics.get('rmse', 0):.4f}, MAE: {metrics.get('mae', 0):.4f}"
            
            print(f"{rank}. {model_name}: 得分={score:.4f}, {summary}")

# ========================
# 数据预处理封装
# ========================
class DataPreprocessor:
    """数据预处理类，封装数据读取和预处理逻辑"""
    
    def __init__(self, control_variables, grouping_vars=None):
        self.control_variables = control_variables
        self.grouping_vars = grouping_vars or []
        self.scalers = {}
        self.imputers = {}
        self.encoders = {}
    
    def load_and_preprocess_data(self, file_path, y_var, t_var):
        """加载并预处理数据"""
        print("步骤1: 数据准备")
        print(f"使用文件: {file_path}")
        
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"文件 '{file_path}' 不存在")
        
        # 读取数据
        data = pd.read_stata(file_path)
        
        # 处理因变量和处理变量
        y_series = data[y_var]
        t_series = data[t_var]
        
        # 编码和预处理
        le = LabelEncoder()
        y = le.fit_transform(y_series) if y_series.dtype == 'object' or isinstance(y_series.dtype, CategoricalDtype) else y_series.values
        t = le.fit_transform(t_series) if t_series.dtype == 'object' or isinstance(t_series.dtype, CategoricalDtype) else t_series.values

        # 更新 data 中的 y_var 和 t_var 列为已编码的值
        data[y_var] = y
        data[t_var] = t

        # 复制分组变量的原始值
        for col in self.grouping_vars:
            if col in data.columns:
                data[f'original_{col}'] = data[col].copy()

        # 检查所有控制变量是否存在于数据中
        missing_cols = [col for col in self.control_variables if col not in data.columns]
        if missing_cols:
            raise ValueError(f"数据中缺少以下控制变量: {missing_cols}")
        
        # 预处理控制变量
        for col in self.control_variables:
            # 处理分类变量
            if data[col].dtype == 'object' or isinstance(data[col].dtype, CategoricalDtype):
                self.encoders[col] = LabelEncoder()
                data[col] = self.encoders[col].fit_transform(data[col].astype(str))
            
            # 处理缺失值
            if data[col].isnull().sum() > 0:
                self.imputers[col] = SimpleImputer(strategy='mean')
                data[col] = self.imputers[col].fit_transform(data[col].values.reshape(-1, 1)).flatten()
            
            # 特征缩放
            self.scalers[col] = StandardScaler()
            data[col] = self.scalers[col].fit_transform(data[col].values.reshape(-1, 1)).flatten()

        # 创建特征矩阵
        X = data[self.control_variables].values
        
        # 转换为GPU数组
        X_gpu = cp.array(X, dtype=cp.float32)
        y_gpu = cp.array(y, dtype=cp.float32)
        t_gpu = cp.array(t, dtype=cp.float32)
        
        # 检查数据完整性
        print("检查数据完整性：")
        print("X 中是否有 NaN 或 Inf:", cp.any(cp.isnan(X_gpu)), cp.any(cp.isinf(X_gpu)))
        print("y 中是否有 NaN 或 Inf:", cp.any(cp.isnan(y_gpu)), cp.any(cp.isinf(y_gpu)))
        print("t 中是否有 NaN 或 Inf:", cp.any(cp.isnan(t_gpu)), cp.any(cp.isinf(t_gpu)))
        
        # 处理缺失值或无穷值
        X_gpu = cp.nan_to_num(X_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        y_gpu = cp.nan_to_num(y_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        t_gpu = cp.nan_to_num(t_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        
        print(f"数据形状: X={X_gpu.shape}, y={y_gpu.shape}, t={t_gpu.shape}")
        
        return cp.asnumpy(X_gpu), cp.asnumpy(y_gpu), cp.asnumpy(t_gpu), data

# ========================
# 新增：异质性分析函数
# ========================
def heterogeneity_analysis(reg_data: pd.DataFrame, var_name: str, groups: List, control_names: List[str], clf_models: List[str]):
    """
    执行异质性分析，对每个分组运行DML分析。
    
    参数:
        reg_data: 数据框
        var_name: 分组变量名称
        groups: 分组规则列表
        control_names: 控制变量列表
        clf_models: 模型名称列表
    
    返回:
        pd.DataFrame: 包含每个组的结果
    """
    results = []
    
    # 确定使用原始列还是预处理列
    var_to_use = f'original_{var_name}' if f'original_{var_name}' in reg_data.columns else var_name
    
    for idx, group in enumerate(groups):
        # 过滤子组数据
        if isinstance(group, list):
            mask = reg_data[var_to_use].isin(group)
            group_label = f"Group {idx+1}: {', '.join(map(str, group))}"
        elif isinstance(group, tuple):
            low, high = group
            mask = (reg_data[var_to_use] > low) & (reg_data[var_to_use] <= high)
            group_label = f"Group {idx+1}: ({low}, {high}]"
        else:
            mask = reg_data[var_to_use] == group
            group_label = f"Group {idx+1}: {group}"
        
        sub_data = reg_data[mask]
        
        if len(sub_data) < 10:  # 跳过样本过小的组
            print(f"跳过 {group_label}: 样本量过小 ({len(sub_data)})")
            continue
        
        # 提取变量
        y = sub_data['entrepreneurship'].values
        t = sub_data['mobile_use'].values
        X = sub_data[control_names].values
        
        # 转换为GPU数组
        X_gpu = cp.array(X, dtype=cp.float32)
        y_gpu = cp.array(y, dtype=cp.float32)
        t_gpu = cp.array(t, dtype=cp.float32)
        
        # 创建模型比较器
        y_comparator = ModelComparator('classification')
        t_comparator = ModelComparator('classification')
        
        # 训练并比较Y模型
        y_models = {}
        y_metrics = {}
        for model_name in clf_models:
            model, metrics = first_stage_cuml_enhanced(X_gpu, y_gpu, task_type='classification', model_name=model_name)
            if model and metrics:
                y_models[model_name] = model
                y_metrics[model_name] = metrics
                y_comparator.add_model(model_name, metrics)
        
        # 选择最佳Y模型
        best_y_model_name = y_comparator.get_best_model()
        if not best_y_model_name:
            print(f"{group_label}: 未能训练Y模型，跳过")
            continue
        best_model_y = y_models[best_y_model_name]
        
        # 训练并比较T模型
        t_models = {}
        t_metrics = {}
        for model_name in clf_models:
            model, metrics = first_stage_cuml_enhanced(X_gpu, t_gpu, task_type='classification', model_name=model_name)
            if model and metrics:
                t_models[model_name] = model
                t_metrics[model_name] = metrics
                t_comparator.add_model(model_name, metrics)
        
        # 选择最佳T模型
        best_t_model_name = t_comparator.get_best_model()
        if not best_t_model_name:
            print(f"{group_label}: 未能训练T模型，跳过")
            continue
        best_model_t = t_models[best_t_model_name]
        
        # 执行DML分析
        dml_res = torch_dml_with_pretrained(
            X, y, t,
            best_model_y,
            best_model_t,
            use_robust_se=True,
            robust_method='bootstrap',
            n_bootstrap=1000,
            n_splits=5
        )
        
        # 收集结果
        results.append({
            'Group': group_label,
            'Effect': dml_res['effect'],
            'SE': dml_res['se'],
            'Lower CI': dml_res['lb'],
            'Upper CI': dml_res['ub'],
            'T-Statistic': dml_res['t_statistic'],
            'P-Value': dml_res['p_value'],
            'F-Statistic': dml_res['f_statistic'],
            'F P-Value': dml_res['f_p_value'],
            'Sample Size': len(sub_data),
            'Best Y Model': best_y_model_name,
            'Best T Model': best_t_model_name,
            'Best Y Metrics': y_metrics[best_y_model_name],
            'Best T Metrics': t_metrics[best_t_model_name]
        })
    
    return pd.DataFrame(results)

# ========================
# 新增：保存详细结果到Word的函数（参考用户提供的格式，但适应当前计算结果）
# ========================
def save_detailed_results(doc, group_label, effect, se, lb, ub, t_statistic, p_value, f_statistic, f_p_value,
                          best_model_y, best_model_t, best_metrics_y, best_metrics_t,
                          robust_method='bootstrap'):
    """
    保存详细结果到Word文档中（参考用户格式，但仅保存当前计算出的结果）。
    注意：当前代码中没有sensitivity_results和batch_id/prefix，因此忽略这些；使用group_label作为标识。
    """
    # 添加段落标题
    doc.add_heading(f"分组: {group_label}", level=2)
    
    # 添加DML结果表格
    doc.add_paragraph("DML 结果:")
    dml_table = doc.add_table(rows=1, cols=2)
    hdr_cells = dml_table.rows[0].cells
    hdr_cells[0].text = '指标'
    hdr_cells[1].text = '值'
    
    dml_data = [
        ('Effect', f"{effect:.6f}"),
        ('SE', f"{se:.6f}"),
        ('Lower CI', f"{lb:.6f}"),
        ('Upper CI', f"{ub:.6f}"),
        ('T-Statistic', f"{t_statistic:.4f}"),
        ('P-Value', f"{p_value:.6f}"),
        ('F-Statistic', f"{f_statistic:.4f}"),
        ('F P-Value', f"{f_p_value:.6f}"),
        ('Robust Method', robust_method)
    ]
    
    for key, val in dml_data:
        row_cells = dml_table.add_row().cells
        row_cells[0].text = key
        row_cells[1].text = val
    
    # 添加最佳模型信息
    doc.add_paragraph(f"最佳 Y 模型: {type(best_model_y).__name__}")
    doc.add_paragraph(f"最佳 T 模型: {type(best_model_t).__name__}")
    
    # 添加Y模型指标
    doc.add_paragraph("最佳 Y 模型指标:")
    y_metrics_table = doc.add_table(rows=1, cols=2)
    hdr_cells = y_metrics_table.rows[0].cells
    hdr_cells[0].text = '指标'
    hdr_cells[1].text = '值'
    
    for key, val in best_metrics_y.items():
        if key != 'predictions' and key != 'model_name' and key != 'task_type':
            row_cells = y_metrics_table.add_row().cells
            row_cells[0].text = key.capitalize()
            row_cells[1].text = f"{val:.4f}" if isinstance(val, (int, float)) else str(val)
    
    # 添加T模型指标
    doc.add_paragraph("最佳 T 模型指标:")
    t_metrics_table = doc.add_table(rows=1, cols=2)
    hdr_cells = t_metrics_table.rows[0].cells
    hdr_cells[0].text = '指标'
    hdr_cells[1].text = '值'
    
    for key, val in best_metrics_t.items():
        if key != 'predictions' and key != 'model_name' and key != 'task_type':
            row_cells = t_metrics_table.add_row().cells
            row_cells[0].text = key.capitalize()
            row_cells[1].text = f"{val:.4f}" if isinstance(val, (int, float)) else str(val)

# ========================
# 主程序逻辑（异质性分析版）
# ========================
if __name__ == "__main__":
    try:
        # 初始化时限制GPU内存
        if torch.cuda.is_available():
            torch.cuda.set_per_process_memory_fraction(0.8)
            
        print("正在执行异质性分析流程")
        print(f"cuML版本: {cuml_version}")
        
        # 定义变量
        file_path = "l3dml_data_digital.dta"
        t_var = 'mobile_use'
        y_var = 'entrepreneurship'
        control_names = [
            'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
            'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize', 
            'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise', 
            'intelligence', 'person_status', 'party', 'workasso',
            'fid','pid','year',
            'ln_total_asset_sq', 'impro_sasti_sq', 'Child_sq',
            'lifefire_sq', 'age_sq', 'ln_renqing_sq', 'cfpsedu_sq',  'familysize_sq', 
            'Oldage_sq', 'health_sq', 'marriage_sq', 'ln_finc_sq', 'exercise_sq', 
            'intelligence_sq', 'person_status_sq', 
            'ln_total_asset_cub', 'impro_sasti_cub', 'Child_cub',
            'lifefire_cub', 'age_cub', 'ln_renqing_cub', 'cfpsedu_cub',  'familysize_cub', 
            'Oldage_cub', 'health_cub', 'marriage_cub', 'ln_finc_cub', 'exercise_cub', 
            'intelligence_cub', 'person_status_cub'
        ]
        
        grouping_vars = ['intelligence', 'health', 'marriage', 'ln_total_asset', 'army', 'laborasso', 'person_status', 'workasso', 'party']
        
        # 定义模型列表
        clf_models = ['RandomForest', 'LogisticRegression', 'SVC', 'XGBoost', 'MLP']
        if HAS_GRADIENT_BOOSTING:
            clf_models.append('GradientBoosting')
        
        # 加载数据
        preprocessor = DataPreprocessor(control_names, grouping_vars=grouping_vars)
        _, _, _, reg_data = preprocessor.load_and_preprocess_data(file_path, y_var, t_var)
        
        # 打印分组变量的唯一值以调试
        print("\n分组变量唯一值：")
        for var in grouping_vars:
            original_var = f'original_{var}' if f'original_{var}' in reg_data.columns else var
            print(f"{var} unique values: {sorted(reg_data[original_var].unique())}")
        
        # 定义分组规则
        intelligence_groups = [[1, 2, 3, 4] ,[5, 6, 7]]
        health_groups = [['一般', '不健康','比较健康'], ['很健康','非常健康']]
        marriage_groups = [['未婚', '离婚', '丧偶', '同居'], [ '在婚（有配偶）']]
        ln_total_asset_groups = [
            (-np.inf, 11.3978),
            (11.3978, 12.2695),
            (12.2695, 12.9739),
            (12.9739, np.inf)
        ]
        army_groups = ['否', '是']
        laborasso_groups = ['否', '是']
        person_status_groups = [[1, 2], [3, 4 ,5]]  # 1 为“低地位”，[2, 3, 4] 为“中等地位”，5 为“高地位”
        workasso_groups = ['否', '是']
        party_groups = ['否', '是']
        
        # 运行异质性分析
        save_path = "./results"
        os.makedirs(save_path, exist_ok=True)
        intelligence_results = heterogeneity_analysis(reg_data, 'intelligence', intelligence_groups, control_names, clf_models)
        health_results = heterogeneity_analysis(reg_data, 'health', health_groups, control_names, clf_models)
        marriage_results = heterogeneity_analysis(reg_data, 'marriage', marriage_groups, control_names, clf_models)
        ln_total_asset_results = heterogeneity_analysis(reg_data, 'ln_total_asset', ln_total_asset_groups, control_names, clf_models)
        army_results = heterogeneity_analysis(reg_data, 'army', army_groups, control_names, clf_models)
        laborasso_results = heterogeneity_analysis(reg_data, 'laborasso', laborasso_groups, control_names, clf_models)
        person_status_results = heterogeneity_analysis(reg_data, 'person_status', person_status_groups, control_names, clf_models)
        workasso_results = heterogeneity_analysis(reg_data, 'workasso', workasso_groups, control_names, clf_models)
        party_results = heterogeneity_analysis(reg_data, 'party', party_groups, control_names, clf_models)
        
        # 保存结果到 CSV
        intelligence_results.to_csv(os.path.join(save_path, "intelligence_heterogeneity.csv"), index=False)
        health_results.to_csv(os.path.join(save_path, "health_heterogeneity.csv"), index=False)
        marriage_results.to_csv(os.path.join(save_path, "marriage_heterogeneity.csv"), index=False)
        ln_total_asset_results.to_csv(os.path.join(save_path, "ln_total_asset_heterogeneity.csv"), index=False)
        army_results.to_csv(os.path.join(save_path, "army_heterogeneity.csv"), index=False)
        laborasso_results.to_csv(os.path.join(save_path, "laborasso_heterogeneity.csv"), index=False)
        person_status_results.to_csv(os.path.join(save_path, "person_status_heterogeneity.csv"), index=False)
        workasso_results.to_csv(os.path.join(save_path, "workasso_heterogeneity.csv"), index=False)
        party_results.to_csv(os.path.join(save_path, "party_heterogeneity.csv"), index=False)
        
        # 创建并填充 Word 文档（使用详细保存函数）
        doc = Document()
        doc.add_heading('异质性分析结果', level=1)
        
        # 为每个结果DataFrame中的每一行调用save_detailed_results
        for _, row in intelligence_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in health_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in marriage_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in ln_total_asset_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in army_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in laborasso_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in person_status_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in workasso_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in party_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        doc.save(os.path.join(save_path, 'heterogeneity_results.docx'))
        print("结果已保存到 Word 文档:", os.path.join(save_path, 'heterogeneity_results.docx'))
        
    except Exception as e:
        print(f"程序执行出错: {str(e)}")
        traceback.print_exc()
        
    finally:
        print("执行终极资源清理...")
        clear_gpu_resources()
        
        if torch.cuda.is_available():
            print("最终GPU内存状态:")
            print(f"已用内存: {torch.cuda.memory_allocated()/1e9:.2f} GB")
            print(f"保留内存: {torch.cuda.memory_reserved()/1e9:.2f} GB")
            
        print("资源清理完成，程序退出！")

/opt/conda/envs/rapids-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuML Gradient Boosting 不可用，将跳过该模型
正在执行异质性分析流程
cuML版本: 25.08.00
步骤1: 数据准备
使用文件: l3dml_data_digital.dta
检查数据完整性：
X 中是否有 NaN 或 Inf: False False
y 中是否有 NaN 或 Inf: False False
t 中是否有 NaN 或 Inf: False False
数据形状: X=(26354, 53), y=(26354,), t=(26354,)

分组变量唯一值：
intelligence unique values: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
health unique values: ['一般', '不健康', '很健康', '比较健康', '非常健康']
marriage unique values: ['丧偶', '同居', '在婚（有配偶）', '未婚', '离婚']
ln_total_asset unique values: [np.float32(0.0), np.float32(3.912023), np.float32(4.6051702), np.float32(5.2983174), np.float32(5.521461), np.float32(5.7037826), np.float32(5.7446046), np.float32(5.857933), np.float32(5.926926), np.float32(6.214608), np.float32(6.3969297), np.float32(6.437752), np.float32(6.802395), np.float32(6.906755), np.float32(6.9077554), np.float32(6.9527287), np.float32(7.0030656), np.float32(7.0255384), np.float32(7.036588), np.float32(7.090077), np.float32(7.130899), np.float32(7.17012), np.float32(7.226209), np.float32(7.261927), 

In [1]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import cupy as cp
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score, confusion_matrix, mean_absolute_error
)
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression

# 修改：正确导入cuML的相关模块
from cuml.ensemble import RandomForestClassifier as cuRF, RandomForestRegressor as cuRFR
from cuml.linear_model import LogisticRegression as cuLogisticRegression, LinearRegression as cuLinearRegression
from cuml.svm import SVC as cuSVC, SVR as cuSVR
from cuml.model_selection import train_test_split as cu_train_test_split  # 修正导入
from cuml.metrics import accuracy_score as cu_accuracy_score
from cuml import __version__ as cuml_version
import warnings
warnings.filterwarnings("ignore", message="Random state is currently ignored by probabilistic SVC")

# 新增：梯度提升模型导入
try:
    from cuml.ensemble import GradientBoostingClassifier as cuGBC, GradientBoostingRegressor as cuGBR
    HAS_GRADIENT_BOOSTING = True
except ImportError:
    print("cuML Gradient Boosting 不可用，将跳过该模型")
    HAS_GRADIENT_BOOSTING = False

# XGBoost导入
import xgboost as xgb

# 其他导入
from datetime import datetime
from pandas.api.types import CategoricalDtype
import traceback
from sklearn.base import clone, BaseEstimator
from sklearn.model_selection import KFold
from typing import Dict, Any, Tuple, Optional, List
import warnings
import json
warnings.filterwarnings('ignore')

# 新增：用于生成Word文档的库
from docx import Document
from docx.shared import Inches

# 新增：用于深度复制PyTorch模型
import copy

# 设置环境变量
os.environ['XGBOOST_DEVICE_MEMORY_LIMIT'] = '10GB'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# ========================
# GPU资源清理增强模块
# ========================
def clear_gpu_resources():
    """全面清理GPU资源并添加延迟"""
    gc.collect()
    time.sleep(0.5)
    
    if 'cp' in globals():
        try:
            cp._default_memory_pool.free_all_blocks()
            cp.get_default_memory_pool().free_all_blocks()
        except Exception: 
            pass
        
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass

def monitor_gpu_memory(threshold=0.8):
    """监控GPU内存使用，超过阈值则清理"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated()
        total = torch.cuda.get_device_properties(0).total_memory
        ratio = allocated / total
        if ratio > threshold:
            print(f"GPU内存使用率超过{threshold*100:.1f}% ({allocated/1e9:.2f}GB/{total/1e9:.2f}GB)，执行清理...")
            clear_gpu_resources()
            return True
    return False

# ========================
# 辅助函数：PyTorch版本的自助法标准误
# ========================
def bootstrap_se_torch(X: np.ndarray, y: np.ndarray, n_bootstrap: int = 1000) -> Tuple[float, float, float]:
    """
    使用自助法计算标准误和置信区间（PyTorch优化版）
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_samples = len(y)
    
    bootstrap_effects = []
    
    for i in range(n_bootstrap):
        # 生成自助样本
        indices = np.random.choice(n_samples, n_samples, replace=True)
        X_boot = torch.tensor(X[indices], dtype=torch.float32, device=device)
        y_boot = torch.tensor(y[indices], dtype=torch.float32, device=device)
        
        # 使用正规方程求解
        with torch.no_grad():
            XtX = X_boot.t() @ X_boot
            Xty = X_boot.t() @ y_boot.reshape(-1, 1)
            
            # 添加小常数防止奇异矩阵
            XtX += torch.eye(X_boot.shape[1], device=device) * 1e-6
            
            coefficients = torch.linalg.solve(XtX, Xty)
            effect = coefficients[1].item() if coefficients.shape[0] > 1 else coefficients[0].item()
        
        bootstrap_effects.append(effect)
    
    # 计算标准误和置信区间
    bootstrap_effects = np.array(bootstrap_effects)
    se = np.std(bootstrap_effects, ddof=1)
    ci_lower = np.percentile(bootstrap_effects, 2.5)
    ci_upper = np.percentile(bootstrap_effects, 97.5)
    
    return se, ci_lower, ci_upper

# ========================
# 辅助函数：PyTorch版本的聚类稳健标准误
# ========================
def calculate_cluster_robust_se_torch(X: torch.Tensor, 
                                     y: torch.Tensor, 
                                     residuals: torch.Tensor, 
                                     cluster_ids: np.ndarray) -> float:
    """
    计算聚类稳健标准误（PyTorch版本）
    """
    device = X.device
    unique_clusters = np.unique(cluster_ids)
    n_clusters = len(unique_clusters)
    
    scores = []
    
    for cluster in unique_clusters:
        cluster_mask = (cluster_ids == cluster)
        X_cluster = X[cluster_mask]
        residuals_cluster = residuals[cluster_mask]
        
        # 计算聚类得分
        cluster_score = X_cluster.t() @ residuals_cluster
        scores.append(cluster_score.cpu().numpy())
    
    # 计算聚类稳健方差估计
    scores_matrix = np.array(scores)
    mean_score = np.mean(scores_matrix, axis=0)
    cluster_variance = (scores_matrix - mean_score).T @ (scores_matrix - mean_score) / (n_clusters - 1)
    
    # 计算标准误
    XtX_inv = torch.linalg.inv(X.t() @ X).cpu().numpy()
    cluster_vcov = XtX_inv @ cluster_variance @ XtX_inv
    se = np.sqrt(np.diag(cluster_vcov))[1]
    
    return se

# ========================
# 新增：第一阶段模型工厂类,修改MLP实现
# ========================
class FirstStageModelFactory:
    """第一阶段模型工厂类，用于创建和配置不同的机器学习模型"""
    def __init__(self):
        self.model_configs = {
            'RandomForest': {
                'classification': lambda: cuRF(n_estimators=200, max_depth=20, random_state=42),
                'regression': lambda: cuRFR(n_estimators=200, max_depth=20, random_state=42)
            },
            'LogisticRegression': {
                'classification': lambda: cuLogisticRegression(penalty='l2', C=1.0, max_iter=1000),  # 移除random_state
                'regression': lambda: cuLinearRegression(fit_intercept=True, algorithm='eig')
            },
            'GradientBoosting': {
                'classification': lambda: cuGBC(n_estimators=200, max_depth=6, random_state=42) if HAS_GRADIENT_BOOSTING else None,
                'regression': lambda: cuGBR(n_estimators=200, max_depth=6, random_state=42) if HAS_GRADIENT_BOOSTING else None
            },
            'SVC': {
                'classification': lambda: cuSVC(kernel='rbf', C=1.0, probability=True, random_state=42),
                'regression': lambda: cuSVR(kernel='rbf', C=1.0)
            },
            'XGBoost': {
                'classification': lambda: xgb.XGBClassifier(n_estimators=200, max_depth=6, random_state=42, tree_method='gpu_hist'),
                'regression': lambda: xgb.XGBRegressor(n_estimators=200, max_depth=6, random_state=42, tree_method='gpu_hist')
            },
            'MLP': {
                'classification': lambda input_size: PyTorchMLP(input_size=input_size, output_size=1, task_type='classification'),
                'regression': lambda input_size: PyTorchMLP(input_size=input_size, output_size=1, task_type='regression')
            }
        }
    
    def get_model(self, model_name, task_type, input_size=None):
        """
        根据模型名称和任务类型获取模型实例
        """
        if model_name not in self.model_configs:
            raise ValueError(f"不支持的模型类型: {model_name}")
        
        if task_type not in self.model_configs[model_name]:
            raise ValueError(f"模型 {model_name} 不支持任务类型: {task_type}")
        
        # 特殊处理MLP模型，需要输入维度
        if model_name == 'MLP':
            if input_size is None:
                raise ValueError("MLP模型需要指定input_size参数")
            return self.model_configs[model_name][task_type](input_size)
        else:
            model = self.model_configs[model_name][task_type]()
            if model is None:
                raise ValueError(f"模型 {model_name} 不可用，可能缺少依赖")
            return model

# ========================
# 新增：PyTorch MLP模型（GPU版本）修改
# ========================
class PyTorchMLP(nn.Module):
    """PyTorch多层感知机模型（GPU加速版），用于替代cuML的MLP"""

    def __init__(self, input_size, hidden_sizes=[100, 50], output_size=1, 
                 task_type='classification', dropout_rate=0.1):
        super(PyTorchMLP, self).__init__()
        self.task_type = task_type
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # 构建网络层
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, output_size))
        
        # 根据任务类型添加输出层激活函数
        if task_type == 'classification':
            layers.append(nn.Sigmoid())
        
        self.network = nn.Sequential(*layers).to(self.device)
        
    def forward(self, x):
        return self.network(x)
    
    def fit(self, X, y, epochs=100, batch_size=256, lr=0.001):
        """训练模型"""
        self.train()
        
        # 转换数据为PyTorch张量
        if not isinstance(X, torch.Tensor):
            X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        else:
            X_tensor = X.to(self.device)
            
        if not isinstance(y, torch.Tensor):
            y_tensor = torch.tensor(y, dtype=torch.float32, device=self.device)
        else:
            y_tensor = y.to(self.device)
        
        # 确保输出维度匹配
        if self.task_type == 'classification':
            y_tensor = y_tensor.view(-1, 1)
        
        # 定义损失函数和优化器
        if self.task_type == 'classification':
            criterion = nn.BCELoss()
        else:
            criterion = nn.MSELoss()
        
        optimizer = optim.Adam(self.parameters(), lr=lr)
        
        # 训练循环
        dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
        
        for epoch in range(epochs):
            epoch_loss = 0.0
            for batch_X, batch_y in dataloader:
                optimizer.zero_grad()
                outputs = self(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            if (epoch + 1) % 20 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(dataloader):.4f}")
    
    def predict(self, X):
        """预测"""
        self.eval()
        with torch.no_grad():
            if not isinstance(X, torch.Tensor):
                X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            else:
                X_tensor = X.to(self.device)
                
            predictions = self(X_tensor)
            
            if self.task_type == 'classification':
                return (predictions > 0.5).float().cpu().numpy().flatten()
            else:
                return predictions.cpu().numpy().flatten()
    
    def predict_proba(self, X):
        """预测概率（仅分类任务）"""
        if self.task_type != 'classification':
            raise ValueError("predict_proba仅适用于分类任务")
        
        self.eval()
        with torch.no_grad():
            if not isinstance(X, torch.Tensor):
                X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            else:
                X_tensor = X.to(self.device)
                
            return self(X_tensor).cpu().numpy()

# ========================
# 新增：使用二折交叉验证计算残差的辅助函数
# ========================
def get_y_residuals(model_proto, X, y, n_splits=5):
    """
    使用交叉验证计算outcome (y) 的残差
    对于y，使用predict (标签) 计算残差
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    residuals = np.zeros(len(y))
    
    for train_idx, test_idx in kf.split(X):
        # 克隆模型
        if isinstance(model_proto, PyTorchMLP):
            model_clone = copy.deepcopy(model_proto)
        else:
            model_clone = clone(model_proto)
        
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # 训练克隆模型
        model_clone.fit(X_train, y_train)
        
        # 预测 (使用predict，得到标签)
        if hasattr(model_clone, 'predict_proba'):
            proba = model_clone.predict_proba(X_test)
            if proba.ndim == 1:
                y_pred = proba
            elif proba.shape[1] == 1:
                y_pred = proba[:, 0]
            else:
                y_pred = proba[:, 1]
        else:
            y_pred = model_clone.predict(X_test)
        
        # 计算残差
        residuals[test_idx] = y_test - y_pred
    
    return residuals

def get_t_residuals(model_proto, X, t, n_splits=5):
    """
    使用交叉验证计算treatment (t) 的残差
    对于t，如果有predict_proba，使用概率；否则使用predict
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    residuals = np.zeros(len(t))
    
    for train_idx, test_idx in kf.split(X):
        # 克隆模型
        if isinstance(model_proto, PyTorchMLP):
            model_clone = copy.deepcopy(model_proto)
        else:
            model_clone = clone(model_proto)
        
        X_train, X_test = X[train_idx], X[test_idx]
        t_train, t_test = t[train_idx], t[test_idx]
        
        # 训练克隆模型
        model_clone.fit(X_train, t_train)
        
        # 预测
        if hasattr(model_clone, 'predict_proba'):
            proba = model_clone.predict_proba(X_test)
            if proba.ndim == 1:
                t_pred = proba
            elif proba.shape[1] == 1:
                t_pred = proba[:, 0]
            else:
                t_pred = proba[:, 1]
        else:
            t_pred = model_clone.predict(X_test)
        
        # 计算残差
        residuals[test_idx] = t_test - t_pred
    
    return residuals

# ========================
# 使用PyTorch进行DML分析（优化版）
# ========================
from scipy import stats
def second_stage_dml_analysis(t_residuals, y_residuals, X_second=None, robust_method='bootstrap', n_bootstrap=1000):
    """
    执行DML的第二阶段分析，计算处理效应及统计检验指标。
    
    参数:
        t_residuals (torch.Tensor或np.array): 第一阶段处理模型的残差。
        y_residuals (torch.Tensor或np.array): 第一阶段结果模型的残差。
        X_second (torch.Tensor或np.array, optional): 用于异质性分析的协变量。默认为None。
        robust_method (str): 稳健标准误方法，'bootstrap'或'hc1'。默认为'bootstrap'。
        n_bootstrap (int): Bootstrap重复次数，仅当robust_method='bootstrap'时有效。默认为1000。
    
    返回:
        dict: 包含效应值、标准误、置信区间、t统计量、p值、F统计量、F检验p值等的字典。
    """
    # 确保将Tensor转换为NumPy数组
    if torch.is_tensor(t_residuals):
        t_residuals = t_residuals.cpu().numpy() if t_residuals.is_cuda else t_residuals.numpy()
    if torch.is_tensor(y_residuals):
        y_residuals = y_residuals.cpu().numpy() if y_residuals.is_cuda else y_residuals.numpy()
    if X_second is not None and torch.is_tensor(X_second):
        X_second = X_second.cpu().numpy() if X_second.is_cuda else X_second.numpy()
    
    n = len(y_residuals)
    
    # 确保残差是NumPy数组并调整为正确的形状
    t_residuals = np.asarray(t_residuals).reshape(-1, 1)
    y_residuals = np.asarray(y_residuals).reshape(-1, 1)
    
    # 第二阶段回归：Y_residuals ~ theta * T_residuals + [其他交互项]
    if X_second is not None:
        # 如果有协变量用于异质性分析（CATE），包含交互项
        X_second = np.asarray(X_second)
        # 构建设计矩阵: [T_residuals, T_residuals * X_second]
        interaction_terms = t_residuals * X_second
        design_matrix = np.column_stack((t_residuals, interaction_terms))
    else:
        # 只估计ATE
        design_matrix = t_residuals
    
    # 使用OLS估计系数
    coefficients, _, _, _ = np.linalg.lstsq(design_matrix, y_residuals, rcond=None)
    
    # 提取处理效应（ATE是第一个系数）
    effect = coefficients[0][0] if len(coefficients) > 0 else 0.0
    
    # 计算预测值和残差
    y_pred = np.dot(design_matrix, coefficients)
    residuals = y_residuals - y_pred
    
    # 初始化变量
    se = 0.0
    t_statistic = 0.0
    p_value = 1.0
    f_statistic = 0.0
    f_p_value = 1.0
    
    # 计算标准误和置信区间
    if robust_method == 'bootstrap':
        bootstrap_effects = []
        for _ in range(n_bootstrap):
            indices = np.random.choice(n, n, replace=True)
            t_boot = t_residuals[indices]
            y_boot = y_residuals[indices]
            if X_second is not None:
                X_second_boot = X_second[indices]
                interaction_boot = t_boot * X_second_boot
                design_boot = np.column_stack((t_boot, interaction_boot))
            else:
                design_boot = t_boot
            coef_boot, _, _, _ = np.linalg.lstsq(design_boot, y_boot, rcond=None)
            bootstrap_effects.append(coef_boot[0][0] if len(coef_boot) > 0 else 0.0)
        se = np.std(bootstrap_effects, ddof=1) if bootstrap_effects else 0.0
    else:
        # 使用HC1稳健标准误的简化实现
        mse = np.sum(residuals**2) / (n - design_matrix.shape[1])
        xtx_inv = np.linalg.inv(np.dot(design_matrix.T, design_matrix))
        se_matrix = mse * xtx_inv
        se = np.sqrt(np.diag(se_matrix)[0]) if se_matrix.size > 0 else 0.0
    
    # 计算置信区间
    ci_lower = effect - 1.96 * se if se > 0 else effect
    ci_upper = effect + 1.96 * se if se > 0 else effect
    
    # 计算t统计量和p值 (只有在se>0时才有意义)
    if se > 0:
        t_statistic = effect / se
        df = n - design_matrix.shape[1]  # 自由度
        p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))
    
    # 计算F统计量和p值 (检验整个第二阶段模型是否显著)
    ss_total = np.sum((y_residuals - np.mean(y_residuals))**2)
    ss_residual = np.sum(residuals**2)
    if ss_residual > 0 and design_matrix.shape[1] > 0:
        ss_explained = ss_total - ss_residual
        df_model = design_matrix.shape[1]
        df_resid = n - df_model
        f_statistic = (ss_explained / df_model) / (ss_residual / df_resid) if df_resid > 0 else 0.0
        f_p_value = 1 - stats.f.cdf(f_statistic, df_model, df_resid) if df_resid > 0 else 1.0
    
    # 返回所有结果
    return {
        'effect': effect,
        'se': se,
        'lb': ci_lower,
        'ub': ci_upper,
        't_statistic': t_statistic,
        'p_value': p_value,
        'f_statistic': f_statistic,
        'f_p_value': f_p_value,
        'df': n - design_matrix.shape[1] if design_matrix.shape[1] > 0 else n
    }

def torch_dml_with_pretrained(X: np.ndarray, 
                              y: np.ndarray, 
                              t: np.ndarray, 
                              model_y: BaseEstimator, 
                              model_t: BaseEstimator, 
                              use_robust_se: bool = True, 
                              robust_method: str = 'bootstrap',
                              n_bootstrap: int = 1000,
                              cluster_ids: Optional[np.ndarray] = None,
                              n_splits: int = 5) -> Dict[str, Any]:
    """
    使用预训练的第一阶段模型执行DML的第二阶段分析（残差回归）
    
    参数:
        X: 协变量矩阵
        y: 结果变量
        t: 处理变量
        model_y: 预训练的结果变量模型
        model_t: 预训练的处理变量模型
        use_robust_se: 是否使用稳健标准误
        robust_method: 稳健标准误计算方法 ('bootstrap' 或 'hc1')
        n_bootstrap: 自助法重采样次数
        cluster_ids: 聚类ID（用于聚类稳健标准误）
        n_splits: 交叉验证折数
    
    返回:
        包含效应值、标准误、置信区间等结果的字典
    """
    print("执行DML第二阶段分析（残差回归）...")
    
    # 初始化所有变量
    effect = 0.0
    se = 0.0
    ci_lower = 0.0
    ci_upper = 0.0
    t_statistic = 0.0
    p_value = 1.0
    f_statistic = 0.0
    f_p_value = 1.0
    df = 0
    
    # 确保输入为NumPy数组
    X_np = X if isinstance(X, np.ndarray) else cp.asnumpy(X)
    y_np = y if isinstance(y, np.ndarray) else cp.asnumpy(y)
    t_np = t if isinstance(t, np.ndarray) else cp.asnumpy(t)
    
    # 第一阶段：使用指定折数交叉验证计算残差
    print(f"使用{n_splits}折交叉验证计算第一阶段残差...")
    y_residuals = get_y_residuals(model_y, X_np, y_np, n_splits=n_splits)
    t_residuals = get_t_residuals(model_t, X_np, t_np, n_splits=n_splits)
    
    # 转换为PyTorch张量
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    t_residuals_tensor = torch.tensor(t_residuals, dtype=torch.float32, device=device).reshape(-1, 1)
    y_residuals_tensor = torch.tensor(y_residuals, dtype=torch.float32, device=device)
    
    # 添加截距项
    ones = torch.ones_like(t_residuals_tensor, device=device)
    X_second = torch.cat([ones, t_residuals_tensor], dim=1)
    
    # 使用正规方程求解第二阶段系数
    with torch.no_grad():
        XtX = X_second.t() @ X_second
        Xty = X_second.t() @ y_residuals_tensor.reshape(-1, 1)
        
        # 添加小常数防止奇异矩阵
        XtX += torch.eye(2, device=device) * 1e-6
        
        coefficients = torch.linalg.solve(XtX, Xty)
        effect = coefficients[1].item()  # 处理效应系数
    
    # 计算标准误和置信区间
    if use_robust_se and robust_method == 'bootstrap':
        print("使用自助法计算稳健标准误...")
        se, ci_lower, ci_upper = bootstrap_se_torch(
            X_second.cpu().numpy(), 
            y_residuals_tensor.cpu().numpy(), 
            n_bootstrap=n_bootstrap
        )
        
        # 计算t统计量和p值
        if se > 0:
            t_statistic = effect / se
            n, k = X_second.shape
            df = n - k
            p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))
    else:
        # 计算经典标准误
        n, k = X_second.shape
        with torch.no_grad():
            y_pred_tensor = X_second @ coefficients
            residuals = y_residuals_tensor.reshape(-1, 1) - y_pred_tensor
        mse = torch.sum(residuals**2) / (n - k)
        cov_matrix = torch.linalg.inv(XtX) * mse
        se = torch.sqrt(torch.diag(cov_matrix))[1].item()
        
        # 计算置信区间
        t_stat_val = 1.96  # 95%置信水平的t统计量
        ci_lower = effect - t_stat_val * se
        ci_upper = effect + t_stat_val * se
        
        # 计算t统计量
        t_statistic = effect / se

        # 计算p值（双侧检验）
        df = n - k  # 自由度
        p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))

        # 计算F统计量
        with torch.no_grad():
            y_pred_tensor = X_second @ coefficients
            ss_residual = torch.sum((y_residuals_tensor.reshape(-1, 1) - y_pred_tensor)**2)
            ss_total = torch.sum((y_residuals_tensor.reshape(-1, 1) - torch.mean(y_residuals_tensor))**2)
            ss_explained = ss_total - ss_residual
    
            # F统计量 = (解释方差/自由度) / (残差方差/自由度)
            f_statistic = (ss_explained / (k - 1)) / (ss_residual / (n - k))
            f_p_value = 1 - stats.f.cdf(f_statistic, k - 1, n - k)
    
    print(f"DML处理效应估计: {effect:.6f}")
    print(f"标准误: {se:.6f}")
    print(f"95%置信区间: [{ci_lower:.6f}, {ci_upper:.6f}]")
    print(f"t统计量: {t_statistic:.4f}")
    print(f"p值: {p_value:.6f}")
    print(f"F统计量: {f_statistic:.4f}")
    print(f"F检验p值: {f_p_value:.6f}")
    
    return {
        'effect': effect,
        'se': se,
        'lb': ci_lower,
        'ub': ci_upper,
        't_statistic': t_statistic,
        'p_value': p_value,
        'f_statistic': f_statistic,
        'f_p_value': f_p_value,
        'df': df
    }

# ========================
# 修改：增强的第一阶段训练函数，尽量使用cuml
# ========================
def first_stage_cuml_enhanced(X, y, task_type='regression', model_name='RandomForest', test_size=0.25):
    """
    增强版第一阶段预测，支持多种模型 - 使用新的评估函数
    """
    clear_gpu_resources()
    
    # 数据分割 - 使用cuML的train_test_split
    X_train, X_test, y_train, y_test = cu_train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    
    # 创建模型
    model_factory = FirstStageModelFactory()
    
    try:
        # 特殊处理MLP模型，需要输入维度
        if model_name == 'MLP':
            input_size = X_train.shape[1]
            model = model_factory.get_model(model_name, task_type, input_size)
        else:
            model = model_factory.get_model(model_name, task_type)
        
        # 特殊处理PyTorch MLP模型
        if model_name == 'MLP':
            # 转换为PyTorch张量
            X_train_pt = torch.tensor(cp.asnumpy(X_train), dtype=torch.float32)
            y_train_pt = torch.tensor(cp.asnumpy(y_train), dtype=torch.float32)
            
            # 训练模型
            model.fit(X_train_pt, y_train_pt)
            
            # 预测
            X_test_pt = torch.tensor(cp.asnumpy(X_test), dtype=torch.float32)
            y_pred = model.predict(X_test_pt)
            y_pred = cp.array(y_pred)
        else:
            # 训练其他cuML/XGBoost模型
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
        
        # 使用新的评估函数计算评估指标 - 添加task_type参数
        metrics = evaluate_model(model, X_test, y_test, model_name, task_type)
        
        # 添加模型信息
        metrics['model_name'] = model_name
        metrics['task_type'] = task_type
        
        # 根据任务类型输出结果
        if task_type == 'classification':
            summary = f"准确率: {metrics.get('accuracy', 0):.4f}, F1: {metrics.get('f1', 0):.4f}, AUC: {metrics.get('auc', 0):.4f}"
        else:
            summary = f"R²: {metrics.get('r2', 0):.4f}, RMSE: {metrics.get('rmse', 0):.4f}, MAE: {metrics.get('mae', 0):.4f}"
        
        print(f"{model_name} {task_type}模型评估 - {summary}")
        
        return model, metrics
        
    except Exception as e:
        print(f"训练{model_name}模型失败: {str(e)}")
        traceback.print_exc()
        return None, None

# ========================
# 新增：评估指标计算函数
# ========================
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
def evaluate_model(model, X_test, y_test, model_name="Model", task_type='classification'):
    """
    评估模型性能，根据任务类型使用不同的指标
    
    参数:
        model: 训练好的模型
        X_test: 测试特征
        y_test: 测试标签
        model_name: 模型名称
        task_type: 任务类型 ('regression' 或 'classification')
    
    返回:
        dict: 包含评估指标的字典
    """
    try:
        # 确保数据是NumPy数组（解决"Implicit conversion to a NumPy array is not allowed"错误）
        if hasattr(X_test, 'get'):
            X_test_np = X_test.get()
        elif hasattr(X_test, 'cpu'):
            X_test_np = X_test.cpu().numpy()
        elif hasattr(X_test, 'numpy'):
            X_test_np = X_test.numpy()
        else:
            X_test_np = np.array(X_test)
            
        if hasattr(y_test, 'get'):
            y_test_np = y_test.get()
        elif hasattr(y_test, 'cpu'):
            y_test_np = y_test.cpu().numpy()
        elif hasattr(y_test, 'numpy'):
            y_test_np = y_test.numpy()
        else:
            y_test_np = np.array(y_test)
        
        # 预测
        y_pred = model.predict(X_test_np)
        
        # 确保预测结果也是NumPy数组
        if hasattr(y_pred, 'get'):
            y_pred_np = y_pred.get()
        elif hasattr(y_pred, 'cpu'):
            y_pred_np = y_pred.cpu().numpy()
        elif hasattr(y_pred, 'numpy'):
            y_pred_np = y_pred.numpy()
        else:
            y_pred_np = np.array(y_pred)
        
        metrics = {}
        
        if task_type == 'regression':
            # 回归任务评估指标
            metrics['r2'] = r2_score(y_test_np, y_pred_np)
            metrics['mse'] = mean_squared_error(y_test_np, y_pred_np)
            metrics['rmse'] = np.sqrt(metrics['mse'])
            metrics['mae'] = mean_absolute_error(y_test_np, y_pred_np)
            
            print(f"{model_name} 回归模型评估 - R²: {metrics['r2']:.4f}, RMSE: {metrics['rmse']:.4f}, MAE: {metrics['mae']:.4f}")
            
        else:
            # 分类任务评估指标
            metrics['accuracy'] = accuracy_score(y_test_np, y_pred_np)
            metrics['precision'] = precision_score(y_test_np, y_pred_np, average='weighted')
            metrics['recall'] = recall_score(y_test_np, y_pred_np, average='weighted')
            metrics['f1'] = f1_score(y_test_np, y_pred_np, average='weighted')
            
            # 计算AUC
            auc_score = np.nan
            try:
                if hasattr(model, 'predict_proba'):
                    y_proba = model.predict_proba(X_test_np)
                    if hasattr(y_proba, 'get'):
                        y_proba = y_proba.get()
                    elif hasattr(y_proba, 'cpu'):
                        y_proba = y_proba.cpu().numpy()
                    
                    if len(y_proba.shape) > 1 and y_proba.shape[1] == 2:  # 二分类
                        auc_score = roc_auc_score(y_test_np, y_proba[:, 1])
                    elif len(y_proba.shape) > 1:  # 多分类
                        auc_score = roc_auc_score(y_test_np, y_proba, multi_class='ovr')
                    else:  # 一维数组
                        auc_score = roc_auc_score(y_test_np, y_proba)
                elif hasattr(model, 'decision_function'):
                    y_score = model.decision_function(X_test_np)
                    if hasattr(y_score, 'get'):
                        y_score = y_score.get()
                    elif hasattr(y_score, 'cpu'):
                        y_score = y_score.cpu().numpy()
                    auc_score = roc_auc_score(y_test_np, y_score)
                else:
                    print(f"警告: 模型 {model_name} 不支持概率预测，无法计算AUC")
                    auc_score = np.nan
            except Exception as e:
                print(f"警告: 计算模型 {model_name} 的AUC时出错: {str(e)}")
                auc_score = np.nan
            
            metrics['auc'] = auc_score
            print(f"{model_name} 分类模型评估 - 准确率: {metrics['accuracy']:.4f}, 精确率: {metrics['precision']:.4f}, 召回率: {metrics['recall']:.4f}, F1: {metrics['f1']:.4f}, AUC: {metrics['auc']:.4f}")
        
        metrics['predictions'] = y_pred_np
        return metrics
        
    except Exception as e:
        print(f"评估模型 {model_name} 时出错: {str(e)}")
        traceback.print_exc()
        if task_type == 'regression':
            return {
                'r2': np.nan,
                'mse': np.nan,
                'rmse': np.nan,
                'mae': np.nan,
                'predictions': None
            }
        else:
            return {
                'accuracy': np.nan,
                'precision': np.nan,
                'recall': np.nan,
                'f1': np.nan,
                'auc': np.nan,
                'predictions': None
            }

# ========================
# 新增：模型比较器类
# ========================
class ModelComparator:
    """模型比较器，用于比较不同模型的性能并选择最佳模型"""
    
    def __init__(self, task_type):
        self.task_type = task_type
        self.model_metrics = {}
        self.weights = self._get_default_weights()
    
    def add_model(self, model_name, metrics):
        """添加模型及其评估指标"""
        self.model_metrics[model_name] = metrics
    
    def _get_default_weights(self):
        """获取默认指标权重"""
        if self.task_type == 'classification':
            return {'accuracy': 0.2, 'precision': 0.2, 'recall': 0.2, 'f1': 0.2, 'auc': 0.2}
        else:
            return {'r2': 0.4, 'rmse': 0.3, 'mae': 0.3}
    
    def _calculate_score(self, metrics):
        """计算模型综合得分（修复版）"""
        if self.task_type == 'regression':
            # 以R²为主要依据，同时考虑RMSE和MAE（数值越小越好，故取倒数）
            r2 = metrics.get('r2', 0)
            rmse = metrics.get('rmse', 1e6)
            mae = metrics.get('mae', 1e6)
            # 如果R²为负，给予惩罚性低分
            if r2 < 0:
                return -1.0 * abs(r2) # 或 return 0.0
            # 综合得分，R²权重最高
            score = r2 * 0.7 + (1 / (rmse + 1e-6)) * 0.15 + (1 / (mae + 1e-6)) * 0.15
            return score
        else:
            # 分类任务：可以考虑准确率、F1、AUC的加权平均
            accuracy = metrics.get('accuracy', 0)
            precision = metrics.get('precision', 0)
            recall = metrics.get('recall', 0)
            f1 = metrics.get('f1', 0)
            auc = metrics.get('auc', 0)
            if np.isnan(auc):
                auc = 0 # 将NaN的AUC视为0
            # 赋予AUC较高权重，如果AUC为0则主要依赖准确率和F1
            if auc > 0:
                score = accuracy * 0.2 + precision * 0.2 + recall * 0.2 + f1 * 0.2 + auc * 0.2
            else:
                score = accuracy * 0.25 + precision * 0.25 + recall * 0.25 + f1 * 0.25
            return score
    
    def compare_models(self):
        """比较所有模型并返回排名"""
        model_scores = {}
        
        for model_name, metrics in self.model_metrics.items():
            score = self._calculate_score(metrics)
            model_scores[model_name] = score
        
        # 按得分排序
        sorted_models = sorted(model_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_models
    
    def get_best_model(self):
        """获取最佳模型名称"""
        sorted_models = self.compare_models()
        return sorted_models[0][0] if sorted_models else None
    
    def print_comparison(self):
        """打印模型比较结果 - 使用新的评估函数"""
        print(f"\n=== {self.task_type.upper()} 模型比较结果 ===")
        sorted_models = self.compare_models()
        
        for rank, (model_name, score) in enumerate(sorted_models, 1):
            metrics = self.model_metrics[model_name]
            
            # 使用新的评估函数格式输出
            if self.task_type == 'classification':
                summary = f"准确率: {metrics.get('accuracy', 0):.4f}, 精确率: {metrics.get('precision', 0):.4f}, 召回率: {metrics.get('recall', 0):.4f}, F1: {metrics.get('f1', 0):.4f}, AUC: {metrics.get('auc', 0):.4f}"
            else:
                summary = f"R²: {metrics.get('r2', 0):.4f}, RMSE: {metrics.get('rmse', 0):.4f}, MAE: {metrics.get('mae', 0):.4f}"
            
            print(f"{rank}. {model_name}: 得分={score:.4f}, {summary}")

# ========================
# 数据预处理封装
# ========================
class DataPreprocessor:
    """数据预处理类，封装数据读取和预处理逻辑"""
    
    def __init__(self, control_variables, grouping_vars=None):
        self.control_variables = control_variables
        self.grouping_vars = grouping_vars or []
        self.scalers = {}
        self.imputers = {}
        self.encoders = {}
    
    def load_and_preprocess_data(self, file_path, y_var, t_var):
        """加载并预处理数据"""
        print("步骤1: 数据准备")
        print(f"使用文件: {file_path}")
        
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"文件 '{file_path}' 不存在")
        
        # 读取数据
        data = pd.read_stata(file_path)
        
        # 处理因变量和处理变量
        y_series = data[y_var]
        t_series = data[t_var]
        
        # 编码和预处理
        le = LabelEncoder()
        y = le.fit_transform(y_series) if y_series.dtype == 'object' or isinstance(y_series.dtype, CategoricalDtype) else y_series.values
        t = le.fit_transform(t_series) if t_series.dtype == 'object' or isinstance(t_series.dtype, CategoricalDtype) else t_series.values

        # 更新 data 中的 y_var 和 t_var 列为已编码的值
        data[y_var] = y
        data[t_var] = t

        # 复制分组变量的原始值
        for col in self.grouping_vars:
            if col in data.columns:
                data[f'original_{col}'] = data[col].copy()

        # 检查所有控制变量是否存在于数据中
        missing_cols = [col for col in self.control_variables if col not in data.columns]
        if missing_cols:
            raise ValueError(f"数据中缺少以下控制变量: {missing_cols}")
        
        # 预处理控制变量
        for col in self.control_variables:
            # 处理分类变量
            if data[col].dtype == 'object' or isinstance(data[col].dtype, CategoricalDtype):
                self.encoders[col] = LabelEncoder()
                data[col] = self.encoders[col].fit_transform(data[col].astype(str))
            
            # 处理缺失值
            if data[col].isnull().sum() > 0:
                self.imputers[col] = SimpleImputer(strategy='mean')
                data[col] = self.imputers[col].fit_transform(data[col].values.reshape(-1, 1)).flatten()
            
            # 特征缩放
            self.scalers[col] = StandardScaler()
            data[col] = self.scalers[col].fit_transform(data[col].values.reshape(-1, 1)).flatten()

        # 创建特征矩阵
        X = data[self.control_variables].values
        
        # 转换为GPU数组
        X_gpu = cp.array(X, dtype=cp.float32)
        y_gpu = cp.array(y, dtype=cp.float32)
        t_gpu = cp.array(t, dtype=cp.float32)
        
        # 检查数据完整性
        print("检查数据完整性：")
        print("X 中是否有 NaN 或 Inf:", cp.any(cp.isnan(X_gpu)), cp.any(cp.isinf(X_gpu)))
        print("y 中是否有 NaN 或 Inf:", cp.any(cp.isnan(y_gpu)), cp.any(cp.isinf(y_gpu)))
        print("t 中是否有 NaN 或 Inf:", cp.any(cp.isnan(t_gpu)), cp.any(cp.isinf(t_gpu)))
        
        # 处理缺失值或无穷值
        X_gpu = cp.nan_to_num(X_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        y_gpu = cp.nan_to_num(y_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        t_gpu = cp.nan_to_num(t_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        
        print(f"数据形状: X={X_gpu.shape}, y={y_gpu.shape}, t={t_gpu.shape}")
        
        return cp.asnumpy(X_gpu), cp.asnumpy(y_gpu), cp.asnumpy(t_gpu), data

# ========================
# 新增：异质性分析函数
# ========================
def heterogeneity_analysis(reg_data: pd.DataFrame, var_name: str, groups: List, control_names: List[str], clf_models: List[str]):
    """
    执行异质性分析，对每个分组运行DML分析。
    
    参数:
        reg_data: 数据框
        var_name: 分组变量名称
        groups: 分组规则列表
        control_names: 控制变量列表
        clf_models: 模型名称列表
    
    返回:
        pd.DataFrame: 包含每个组的结果
    """
    results = []
    
    # 确定使用原始列还是预处理列
    var_to_use = f'original_{var_name}' if f'original_{var_name}' in reg_data.columns else var_name
    
    for idx, group in enumerate(groups):
        # 过滤子组数据
        if isinstance(group, list):
            mask = reg_data[var_to_use].isin(group)
            group_label = f"Group {idx+1}: {', '.join(map(str, group))}"
        elif isinstance(group, tuple):
            low, high = group
            mask = (reg_data[var_to_use] > low) & (reg_data[var_to_use] <= high)
            group_label = f"Group {idx+1}: ({low}, {high}]"
        else:
            mask = reg_data[var_to_use] == group
            group_label = f"Group {idx+1}: {group}"
        
        sub_data = reg_data[mask]
        
        if len(sub_data) < 10:  # 跳过样本过小的组
            print(f"跳过 {group_label}: 样本量过小 ({len(sub_data)})")
            continue
        
        if np.var(sub_data['entrepreneurship']) == 0 or np.var(sub_data['mobile_use']) == 0:
            print(f"跳过 {group_label}: 变量无方差")
            continue
        
        # 提取变量
        y = sub_data['entrepreneurship'].values
        t = sub_data['mobile_use'].values
        X = sub_data[control_names].values
        
        # 转换为GPU数组
        X_gpu = cp.array(X, dtype=cp.float32)
        y_gpu = cp.array(y, dtype=cp.float32)
        t_gpu = cp.array(t, dtype=cp.float32)
        
        # 创建模型比较器
        y_comparator = ModelComparator('classification')
        t_comparator = ModelComparator('classification')
        
        # 训练并比较Y模型
        y_models = {}
        y_metrics = {}
        for model_name in clf_models:
            model, metrics = first_stage_cuml_enhanced(X_gpu, y_gpu, task_type='classification', model_name=model_name)
            if model and metrics:
                y_models[model_name] = model
                y_metrics[model_name] = metrics
                y_comparator.add_model(model_name, metrics)
        
        # 选择最佳Y模型
        best_y_model_name = y_comparator.get_best_model()
        if not best_y_model_name:
            print(f"{group_label}: 未能训练Y模型，跳过")
            continue
        best_model_y = y_models[best_y_model_name]
        
        # 训练并比较T模型
        t_models = {}
        t_metrics = {}
        for model_name in clf_models:
            model, metrics = first_stage_cuml_enhanced(X_gpu, t_gpu, task_type='classification', model_name=model_name)
            if model and metrics:
                t_models[model_name] = model
                t_metrics[model_name] = metrics
                t_comparator.add_model(model_name, metrics)
        
        # 选择最佳T模型
        best_t_model_name = t_comparator.get_best_model()
        if not best_t_model_name:
            print(f"{group_label}: 未能训练T模型，跳过")
            continue
        best_model_t = t_models[best_t_model_name]
        
        # 执行DML分析
        dml_res = torch_dml_with_pretrained(
            X, y, t,
            best_model_y,
            best_model_t,
            use_robust_se=True,
            robust_method='bootstrap',
            n_bootstrap=1000,
            n_splits=5
        )
        
        # 收集结果
        results.append({
            'Group': group_label,
            'Effect': dml_res['effect'],
            'SE': dml_res['se'],
            'Lower CI': dml_res['lb'],
            'Upper CI': dml_res['ub'],
            'T-Statistic': dml_res['t_statistic'],
            'P-Value': dml_res['p_value'],
            'F-Statistic': dml_res['f_statistic'],
            'F P-Value': dml_res['f_p_value'],
            'Sample Size': len(sub_data),
            'Best Y Model': best_y_model_name,
            'Best T Model': best_t_model_name,
            'Best Y Metrics': y_metrics[best_y_model_name],
            'Best T Metrics': t_metrics[best_t_model_name]
        })
    
    return pd.DataFrame(results)

# ========================
# 新增：保存详细结果到Word的函数（参考用户提供的格式，但适应当前计算结果）
# ========================
def save_detailed_results(doc, group_label, effect, se, lb, ub, t_statistic, p_value, f_statistic, f_p_value,
                          best_model_y, best_model_t, best_metrics_y, best_metrics_t,
                          robust_method='bootstrap'):
    """
    保存详细结果到Word文档中（参考用户格式，但仅保存当前计算出的结果）。
    注意：当前代码中没有sensitivity_results和batch_id/prefix，因此忽略这些；使用group_label作为标识。
    """
    # 添加段落标题
    doc.add_heading(f"分组: {group_label}", level=2)
    
    # 添加DML结果表格
    doc.add_paragraph("DML 结果:")
    dml_table = doc.add_table(rows=1, cols=2)
    hdr_cells = dml_table.rows[0].cells
    hdr_cells[0].text = '指标'
    hdr_cells[1].text = '值'
    
    dml_data = [
        ('Effect', f"{effect:.6f}"),
        ('SE', f"{se:.6f}"),
        ('Lower CI', f"{lb:.6f}"),
        ('Upper CI', f"{ub:.6f}"),
        ('T-Statistic', f"{t_statistic:.4f}"),
        ('P-Value', f"{p_value:.6f}"),
        ('F-Statistic', f"{f_statistic:.4f}"),
        ('F P-Value', f"{f_p_value:.6f}"),
        ('Robust Method', robust_method)
    ]
    
    for key, val in dml_data:
        row_cells = dml_table.add_row().cells
        row_cells[0].text = key
        row_cells[1].text = val
    
    # 添加最佳模型信息
    doc.add_paragraph(f"最佳 Y 模型: {type(best_model_y).__name__}")
    doc.add_paragraph(f"最佳 T 模型: {type(best_model_t).__name__}")
    
    # 添加Y模型指标
    doc.add_paragraph("最佳 Y 模型指标:")
    y_metrics_table = doc.add_table(rows=1, cols=2)
    hdr_cells = y_metrics_table.rows[0].cells
    hdr_cells[0].text = '指标'
    hdr_cells[1].text = '值'
    
    for key, val in best_metrics_y.items():
        if key != 'predictions' and key != 'model_name' and key != 'task_type':
            row_cells = y_metrics_table.add_row().cells
            row_cells[0].text = key.capitalize()
            row_cells[1].text = f"{val:.4f}" if isinstance(val, (int, float)) else str(val)
    
    # 添加T模型指标
    doc.add_paragraph("最佳 T 模型指标:")
    t_metrics_table = doc.add_table(rows=1, cols=2)
    hdr_cells = t_metrics_table.rows[0].cells
    hdr_cells[0].text = '指标'
    hdr_cells[1].text = '值'
    
    for key, val in best_metrics_t.items():
        if key != 'predictions' and key != 'model_name' and key != 'task_type':
            row_cells = t_metrics_table.add_row().cells
            row_cells[0].text = key.capitalize()
            row_cells[1].text = f"{val:.4f}" if isinstance(val, (int, float)) else str(val)

# ========================
# 主程序逻辑（异质性分析版）
# ========================
if __name__ == "__main__":
    try:
        # 初始化时限制GPU内存
        if torch.cuda.is_available():
            torch.cuda.set_per_process_memory_fraction(0.8)
            
        print("正在执行异质性分析流程")
        print(f"cuML版本: {cuml_version}")
        
        # 定义变量
        file_path = "l3dml_data_digital.dta"
        t_var = 'mobile_use'
        y_var = 'entrepreneurship'
        control_names = [
            'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
            'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize', 
            'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise', 
            'intelligence', 'person_status', 'party', 'workasso',
            'fid','pid','year',
            'ln_total_asset_sq', 'impro_sasti_sq', 'Child_sq',
            'lifefire_sq', 'age_sq', 'ln_renqing_sq', 'cfpsedu_sq',  'familysize_sq', 
            'Oldage_sq', 'health_sq', 'marriage_sq', 'ln_finc_sq', 'exercise_sq', 
            'intelligence_sq', 'person_status_sq', 
            'ln_total_asset_cub', 'impro_sasti_cub', 'Child_cub',
            'lifefire_cub', 'age_cub', 'ln_renqing_cub', 'cfpsedu_cub',  'familysize_cub', 
            'Oldage_cub', 'health_cub', 'marriage_cub', 'ln_finc_cub', 'exercise_cub', 
            'intelligence_cub', 'person_status_cub'
        ]
        
        grouping_vars = ['intelligence', 'health', 'marriage', 'ln_total_asset', 'army', 'laborasso', 'person_status', 'workasso', 'party']
        
        # 定义模型列表
        clf_models = ['RandomForest', 'LogisticRegression', 'SVC', 'XGBoost', 'MLP']
        if HAS_GRADIENT_BOOSTING:
            clf_models.append('GradientBoosting')
        
        # 加载数据
        preprocessor = DataPreprocessor(control_names, grouping_vars=grouping_vars)
        _, _, _, reg_data = preprocessor.load_and_preprocess_data(file_path, y_var, t_var)
        
        # 打印分组变量的唯一值以调试
        print("\n分组变量唯一值：")
        for var in grouping_vars:
            original_var = f'original_{var}' if f'original_{var}' in reg_data.columns else var
            print(f"{var} unique values: {sorted(reg_data[original_var].unique())}")
        
        # 定义分组规则
        intelligence_groups = [[1, 2, 3, 4] ,[5, 6, 7]]
        health_groups = [['一般', '不健康','比较健康'], ['很健康','非常健康']]
        marriage_groups = [['未婚', '离婚', '丧偶', '同居'], [ '在婚（有配偶）']]
        ln_total_asset_groups = [
            (-np.inf, 11.3978),
            (11.3978, 12.2695),
            (12.2695, 12.9739),
            (12.9739, np.inf)
        ]
        army_groups = ['否', '是']
        laborasso_groups = ['否', '是']
        person_status_groups = [[1, 2], [3, 4 ,5]]  # 1 为“低地位”，[2, 3, 4] 为“中等地位”，5 为“高地位”
        workasso_groups = ['否', '是']
        party_groups = ['否', '是']
        
        # 运行异质性分析
        save_path = "./results"
        os.makedirs(save_path, exist_ok=True)
        intelligence_results = heterogeneity_analysis(reg_data, 'intelligence', intelligence_groups, control_names, clf_models)
        health_results = heterogeneity_analysis(reg_data, 'health', health_groups, control_names, clf_models)
        marriage_results = heterogeneity_analysis(reg_data, 'marriage', marriage_groups, control_names, clf_models)
        ln_total_asset_results = heterogeneity_analysis(reg_data, 'ln_total_asset', ln_total_asset_groups, control_names, clf_models)
        army_results = heterogeneity_analysis(reg_data, 'army', army_groups, control_names, clf_models)
        laborasso_results = heterogeneity_analysis(reg_data, 'laborasso', laborasso_groups, control_names, clf_models)
        person_status_results = heterogeneity_analysis(reg_data, 'person_status', person_status_groups, control_names, clf_models)
        workasso_results = heterogeneity_analysis(reg_data, 'workasso', workasso_groups, control_names, clf_models)
        party_results = heterogeneity_analysis(reg_data, 'party', party_groups, control_names, clf_models)
        
        # 保存结果到 CSV
        intelligence_results.to_csv(os.path.join(save_path, "intelligence_heterogeneity.csv"), index=False)
        health_results.to_csv(os.path.join(save_path, "health_heterogeneity.csv"), index=False)
        marriage_results.to_csv(os.path.join(save_path, "marriage_heterogeneity.csv"), index=False)
        ln_total_asset_results.to_csv(os.path.join(save_path, "ln_total_asset_heterogeneity.csv"), index=False)
        army_results.to_csv(os.path.join(save_path, "army_heterogeneity.csv"), index=False)
        laborasso_results.to_csv(os.path.join(save_path, "laborasso_heterogeneity.csv"), index=False)
        person_status_results.to_csv(os.path.join(save_path, "person_status_heterogeneity.csv"), index=False)
        workasso_results.to_csv(os.path.join(save_path, "workasso_heterogeneity.csv"), index=False)
        party_results.to_csv(os.path.join(save_path, "party_heterogeneity.csv"), index=False)
        
        # 创建并填充 Word 文档（使用详细保存函数）
        doc = Document()
        doc.add_heading('异质性分析结果', level=1)
        
        # 为每个结果DataFrame中的每一行调用save_detailed_results
        for _, row in intelligence_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in health_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in marriage_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in ln_total_asset_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in army_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in laborasso_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in person_status_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in workasso_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        for _, row in party_results.iterrows():
            save_detailed_results(doc, row['Group'], row['Effect'], row['SE'], row['Lower CI'], row['Upper CI'],
                                  row['T-Statistic'], row['P-Value'], row['F-Statistic'], row['F P-Value'],
                                  row['Best Y Model'], row['Best T Model'], row['Best Y Metrics'], row['Best T Metrics'])
        
        doc.save(os.path.join(save_path, 'heterogeneity_results.docx'))
        print("结果已保存到 Word 文档:", os.path.join(save_path, 'heterogeneity_results.docx'))
        
    except Exception as e:
        print(f"程序执行出错: {str(e)}")
        traceback.print_exc()
        
    finally:
        print("执行终极资源清理...")
        clear_gpu_resources()
        
        if torch.cuda.is_available():
            print("最终GPU内存状态:")
            print(f"已用内存: {torch.cuda.memory_allocated()/1e9:.2f} GB")
            print(f"保留内存: {torch.cuda.memory_reserved()/1e9:.2f} GB")
            
        print("资源清理完成，程序退出！")

/opt/conda/envs/rapids-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuML Gradient Boosting 不可用，将跳过该模型
正在执行异质性分析流程
cuML版本: 25.08.00
步骤1: 数据准备
使用文件: l3dml_data_digital.dta
检查数据完整性：
X 中是否有 NaN 或 Inf: False False
y 中是否有 NaN 或 Inf: False False
t 中是否有 NaN 或 Inf: False False
数据形状: X=(26354, 53), y=(26354,), t=(26354,)

分组变量唯一值：
intelligence unique values: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
health unique values: ['一般', '不健康', '很健康', '比较健康', '非常健康']
marriage unique values: ['丧偶', '同居', '在婚（有配偶）', '未婚', '离婚']
ln_total_asset unique values: [np.float32(0.0), np.float32(3.912023), np.float32(4.6051702), np.float32(5.2983174), np.float32(5.521461), np.float32(5.7037826), np.float32(5.7446046), np.float32(5.857933), np.float32(5.926926), np.float32(6.214608), np.float32(6.3969297), np.float32(6.437752), np.float32(6.802395), np.float32(6.906755), np.float32(6.9077554), np.float32(6.9527287), np.float32(7.0030656), np.float32(7.0255384), np.float32(7.036588), np.float32(7.090077), np.float32(7.130899), np.float32(7.17012), np.float32(7.226209), np.float32(7.261927), 